# Kitsune: online packet-header classification

The classifier is **online**: its input is each packet's header fields, in
order, plus the time since the previous packet of the flow, and it can decide after every packet. Only the PCAPs and
label files are used.

This notebook holds the whole process and is meant to be run top to bottom in one kernel. Shared code is in
`kitsune_lib.py`.

| Stage | Contents |
|---|---|
| 0 | data, PCAP parsing, flows, 50-packet segments, inputs |
| 1 | diagnostics before the split, Tracks A and B, time split |
| 2 | GRU size; main model (all feature groups) and model without TTL, destination port and IAT |
| 3 | deployable units, thresholds, all/unseen views, ceilings, baselines, per attack type, held-out traffic, confidence intervals, drift, evasion, cost |
| 4 | retraining on OOD, unseen sessions, whole flows |
| 5 | stateful GRU: memory carried across a flow's segments, against the main model |
| 6 | MC dropout on the chosen model (Track A, without the testbed group): aleatoric and epistemic uncertainty |
| 7 | drift monitoring with the population stability index (PSI) |

Stages 2-5 run for Tracks A and B, stages 6 and 7 for Track A only. Before each track the namespace is cleared and
`TRACK` is set, so every section runs on its own track's data only.

**Tracks.** A: labels as recorded. B: conflicting sequences removed, the best case after manual review (not every
conflict may be resolvable). Results on real data, where conflicting labels are addressed when possible, are expected to fall between the two.

**Reproducibility.** The package versions in `src/Project/data/results/environment.txt` (Python 3.13.7, torch 2.9.0+cpu
and the full freeze listed there) are required to reproduce the results; install them with
`pip install -r src/Project/data/results/environment.txt` (lines starting with `#` are comments). Every random step uses
`SEED = 0`; GRU training gives bit-identical weights on rerun with the same package versions and the same number of
CPU threads.

Models and intermediate results are cached (`data/models_v2`, `data/results`, `data/tracks`), so a rerun reuses the
trained models; delete them to recompute from scratch.

In [ ]:
# Setup: every stage runs from the project folder src/Project (where data/ and kitsune_lib.py are), wherever this
# notebook is opened. %reset in the stages below clears variables only, so the folder and the import path stay set.
import os
import sys
from pathlib import Path

PROJECT = next((p / "src" / "Project" for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "Project" / "kitsune_lib.py").exists()),
               None) or (Path.cwd() if (Path.cwd() / "kitsune_lib.py").exists() else None)
if PROJECT is None:
    raise FileNotFoundError("src/Project not found above " + str(Path.cwd()) + ": open the notebook inside the repository")
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print("project folder:", PROJECT)

In [ ]:
import gzip
import time
import zipfile
from pathlib import Path

import dpkt
import numpy as np
import pandas as pd

import kitsune_lib
from kitsune_lib import CAPTURES, COL, COLUMNS, L, OOD_CAPTURES, SEED  # noqa: F401  (used further down)

# 9 attack folders, each with <Attack>_pcap, _labels and _dataset (unused). 19 GB zipped.
DATA_DIR = Path("data")
archive = zipfile.ZipFile(DATA_DIR / "kitsune+network+attack+dataset.zip")
for m in archive.infolist():
    if "_pcap" in m.filename or "_labels" in m.filename:
        print(f"{m.file_size / 1e6:8.1f} MB  {m.filename}")

In [ ]:
# Data exploration: what the raw data looks like, using Mirai as the example.
# Each capture is a gzipped PCAP plus a labels CSV holding one 0/1 label per packet, in packet order.
# (member() is defined further down, so the files are looked up directly here.)
pcap_name = next(n for n in archive.namelist() if n.startswith("mirai/") and "_pcap" in n)
labels_name = next(n for n in archive.namelist() if n.startswith("mirai/") and "_labels" in n)

with archive.open(labels_name) as f, gzip.open(f) as g:
    labels = pd.read_csv(g, header=None, names=["label"])["label"]
first_attack = int(labels.to_numpy().argmax())
print(f"{labels_name}: {len(labels):,} rows, no header, one label per packet")
print(f"  benign (0): {(labels == 0).sum():,}   malicious (1): {(labels == 1).sum():,}")
print(f"  first malicious packet: number {first_attack + 1:,}\n")


def describe(ts, buf):
    eth = dpkt.ethernet.Ethernet(buf)
    row = {"time": ts, "bytes": len(buf), "src": dpkt.utils.mac_to_str(eth.src),
           "dst": dpkt.utils.mac_to_str(eth.dst), "protocol": type(eth.data).__name__, "ports": ""}
    if isinstance(eth.data, bytes):  # EtherType dpkt does not decode
        row["protocol"] = f"type 0x{eth.type:04x}"
    if isinstance(eth.data, (dpkt.ip.IP, dpkt.ip6.IP6)):
        ip = eth.data
        row["src"], row["dst"] = dpkt.utils.inet_to_str(ip.src), dpkt.utils.inet_to_str(ip.dst)
        row["protocol"] = type(ip.data).__name__
        if isinstance(ip.data, (dpkt.tcp.TCP, dpkt.udp.UDP)):
            row["ports"] = f"{ip.data.sport} -> {ip.data.dport}"
    return row


# Despite its .pcap extension, Mirai's file is pcapng.
shown = set(range(5)) | set(range(first_attack - 2, first_attack + 3))
rows = []
with archive.open(pcap_name) as f, gzip.open(f) as g:
    for i, (ts, buf) in enumerate(dpkt.pcapng.Reader(g)):
        if i in shown:
            rows.append({"packet": i + 1, **describe(ts, buf), "label": labels[i]})
        if i == first_attack:
            attack_buf = buf
        if i > max(shown):
            break
packets = pd.DataFrame(rows)
packets["time"] = (packets["time"] - packets["time"].iloc[0]).round(3)  # seconds since the first packet
print(f"{pcap_name}: first packets, then the packets around the first malicious one")
print(packets.to_string(index=False))

# The table above is a selection. Every header field of one packet, layer by layer:
print(f"\nAll fields of packet {first_attack + 1:,} (the first malicious one), {len(attack_buf)} bytes captured:")
layer = dpkt.ethernet.Ethernet(attack_buf)
while isinstance(layer, dpkt.Packet):
    fields = {}
    for name in layer.__hdr_fields__:
        # Fields packed into shared bytes (e.g. IP version + header length) are split into their parts
        for part in [p for p, _ in getattr(layer, "__bit_fields__", {}).get(name, [(name, 0)]) if not p.startswith("_")]:
            value = getattr(layer, part)
            if part in ("src", "dst"):
                value = dpkt.utils.mac_to_str(value) if len(value) == 6 else dpkt.utils.inet_to_str(value)
            fields[part] = value
    print(f"  {type(layer).__name__:<9}" + ", ".join(f"{k}={v}" for k, v in fields.items()))
    layer = layer.data
print(f"  {'payload':<9}{len(layer)} bytes: {layer!r}")

In [ ]:
# Capture -> folder and role (kitsune_lib.CAPTURES); ARP MitM and Mirai are held out
# (kitsune_lib.OOD_CAPTURES). Segments of L = 50 packets; SEED = 0 for every random step.
PACKET_DIR = DATA_DIR / "packets"
PACKET_DIR.mkdir(exist_ok=True)


def member(capture, suffix):
    return next(n for n in archive.namelist() if n.startswith(CAPTURES[capture] + "/") and suffix in n)


def read_labels(capture):
    # Mirai's file is a bare column; the others have a header and an index column.
    with archive.open(member(capture, "_labels")) as f, gzip.open(f) as g:
        labels = pd.read_csv(g, header=None, low_memory=False).iloc[:, -1]
    return pd.to_numeric(labels, errors="coerce").dropna().to_numpy(dtype=np.int8)


def parse_capture(capture):
    """One row per packet, in file order so it lines up with the labels file, parsed by kitsune_lib.parse_pcap (the
    parser infer.py uses for new captures)."""
    labels = read_labels(capture)
    with archive.open(member(capture, "_pcap")) as f, gzip.open(f) as g:
        out = kitsune_lib.parse_pcap(g)
    assert len(out) == len(labels), f"{capture}: {len(out):,} packets but {len(labels):,} labels"
    out["label"] = labels
    return out

In [ ]:
# Parse every PCAP with dpkt (~4 min); cached after the first run.
# parse_capture asserts that the packet count equals the label count, capture by capture.
t0 = time.time()
for capture, folder in CAPTURES.items():
    path = PACKET_DIR / f"{folder}.parquet"
    if not path.exists():
        parse_capture(capture).to_parquet(path, index=False)
        print(f"{capture}: parsed, {time.time() - t0:.0f}s")

overview = []
for capture, folder in CAPTURES.items():
    p = pd.read_parquet(PACKET_DIR / f"{folder}.parquet", columns=["ts", "flow", "label", "ip_proto", "ethertype"])
    overview.append({
        "capture": capture, "packets": len(p), "malicious share": p["label"].mean(),
        "minutes": (p["ts"].max() - p["ts"].min()) / 60,
        "attack starts at (share of capture)": p["label"].to_numpy().argmax() / len(p),
        "TCP": (p["ip_proto"] == 6).mean(), "UDP": (p["ip_proto"] == 17).mean(),
        "ARP": (p["ethertype"] == 0x806).mean(),
    })
overview = pd.DataFrame(overview).set_index("capture")
overview.round(3)

In [ ]:
# What the parsed packets look like: the first packets of one capture, and the first label change
p = pd.read_parquet(PACKET_DIR / "active_wiretap.parquet")
p.insert(0, "packet", np.arange(1, len(p) + 1))
shown_cols = ["packet", "ts", "flow", "direction", "frame_len", "ethertype", "ip_proto", "ttl", "l4_hlen", "tcp_flags", "tcp_window", "dst_port", "label"]
print(f"Active Wiretap: {p.shape[0]:,} packets, {p.shape[1] - 1} fields. First packets:")
print(p[shown_cols].head(5).to_string(index=False))
first_attack = int(p["label"].to_numpy().argmax())
print(f"\nFirst malicious packet is number {first_attack + 1:,}. Around it:")
print(p[shown_cols].iloc[first_attack - 2:first_attack + 3].to_string(index=False))
print("\nProtocols (ip_proto):", p["ip_proto"].value_counts().head(5).to_dict())

## Flows, segments and inputs

Over all nine captures:

1. **Flows.** IP packets are grouped by 5-tuple, non-IP packets by MAC pair and EtherType, both directions together.
2. **Flow segments.** Each flow is cut into consecutive segments of L = 50 packets and **each segment is one sample**.
   The table below shows why: whole flows, or only their first L packets, leave few samples and most packets unseen.
   A segment is labelled attack if any of its packets is labelled malicious. The classifier starts fresh on each, so
   long flows are re-checked every L packets (Stage 5 carries memory across them instead).
3. **Inputs.** Header fields plus the inter-arrival time (IAT). Addresses group packets into flows and are then dropped.
   TTL, destination port and IAT are inputs of the main model; because they may identify this testbed (hosts,
   services, network timing) rather than the attack, a second model is trained without them (Stage 2).
4. **Diagnostics, tracks and the split** are in Stage 1 below.

In [ ]:
# Why segments: whole flows vs L-packet segments. A sample is attack if any of its packets is malicious.
comparison = []
for capture, folder in CAPTURES.items():
    p = pd.read_parquet(PACKET_DIR / f"{folder}.parquet", columns=["ts", "flow", "label"])
    p = p.sort_values(["flow", "ts"], kind="stable")
    seq = p.groupby("flow").cumcount()
    whole = p.groupby("flow")["label"].max()
    first_l = p[seq < L].groupby("flow")["label"].max()  # label seen by a first-L-packets model
    segment = p.groupby(["flow", seq // L])["label"].max()
    comparison.append({
        "capture": capture, "flows": len(whole), "flows, benign": int((whole == 0).sum()),
        "flows, attack": int(whole.sum()), "largest flow, packets": int(p.groupby("flow").size().max()),
        "packets in first L": int((seq < L).sum()), "attack flows benign in first L": int(whole.sum() - first_l.sum()),
        "segments": len(segment), "segments, benign": int((segment == 0).sum()), "segments, attack": int(segment.sum()),
    })
comparison = pd.DataFrame(comparison).set_index("capture")
comparison.to_csv(DATA_DIR / "flows_vs_segments.csv")
print("in-distribution totals:", comparison.drop(OOD_CAPTURES)[["flows", "flows, benign", "segments", "segments, benign"]].sum().to_dict())
comparison

Reading the table: a capture is a few cameras holding a few very long connections, plus, in OS Scan and SSDP Flood, tens of thousands of tiny attack flows. Whole flows give only a few hundred benign samples and single-digit attack samples in some captures, and most packets are never seen, so a per-capture 60/20/20 split is not possible. An idle timeout does not help because streams rarely pause. Segments keep nearly all traffic and keep the online logic unchanged.

In [ ]:
# Per-packet inputs: kitsune_lib.COLUMNS (sizes, header lengths, TCP flags and window, protocol, direction, IAT, TTL,
# destination port, and the mask of real packets), built by kitsune_lib.packet_features, the same function infer.py
# uses. Continuous columns are log-scaled there and standardized with training statistics later.
# What makes two packets "the same": two keys, each holding exactly what one model reads (kitsune_lib.packet_keys).
# "headers": every header field except TTL and destination port (the model without TTL, destination port and IAT).
# "all inputs": the same plus TTL, destination port and the IAT in whole microseconds (the main model). The label is
# never part of a key. Stage 1 reports duplicates and conflicts under both.


SEQ_FILE, SEG_FILE, HASH_FILE = DATA_DIR / "sequences.npy", DATA_DIR / "segments.parquet", DATA_DIR / "packet_hashes.npz"
PREFIX_FILE = DATA_DIR / "prefix_hashes.npz"
if not all(f.exists() for f in (SEQ_FILE, SEG_FILE, HASH_FILE, PREFIX_FILE)):
    t0 = time.time()
    n_total = int(comparison["segments"].sum())
    X_out = np.lib.format.open_memmap(SEQ_FILE, mode="w+", dtype=np.float32, shape=(n_total, L, len(COLUMNS)))
    H_header, H_all = np.zeros((n_total, L), np.uint64), np.zeros((n_total, L), np.uint64)
    meta, offset, iat_rows = [], 0, []
    for capture, folder in CAPTURES.items():
        p = pd.read_parquet(PACKET_DIR / f"{folder}.parquet").sort_values(["flow", "ts"], kind="stable").reset_index(drop=True)
        pos, seg = kitsune_lib.flow_positions(p, L)
        seg = seg + offset
        kitsune_lib.add_protocol(p)
        gap_us = kitsune_lib.packet_iat(p, pos) * 1e6
        iat_us = np.rint(gap_us).astype(np.int64)
        X_out[seg, pos] = kitsune_lib.packet_features(p, pos)
        H_header[seg, pos], H_all[seg, pos] = kitsune_lib.packet_keys(p, iat_us)
        # The "all inputs" key holds the IAT in whole microseconds; the model reads the exact gap. Record how many gaps
        # are not whole microseconds, i.e. where the key is coarser than what the model reads.
        iat_rows.append({"capture": capture, "packets": len(p),
                         "gaps not whole microseconds": int((np.abs(gap_us - np.rint(gap_us)) > 1e-3).sum())})
        m = pd.DataFrame({"segment": seg, "label": p["label"], "ts": p["ts"], "is_arp": p["ethertype"] == 0x806})
        m = m.groupby("segment").agg(attack=("label", "max"), malicious=("label", "sum"), length=("label", "size"),
                                     start=("ts", "min"), arp_share=("is_arp", "mean")).reset_index()
        m["capture"] = capture
        meta.append(m)
        offset = seg[-1] + 1
        print(f"\r{capture:18} {time.time() - t0:4.0f}s", end="")
    assert offset == n_total
    X_out.flush()
    del X_out
    pd.concat(meta, ignore_index=True).to_parquet(SEG_FILE, index=False)
    np.savez(HASH_FILE, header=H_header, all_inputs=H_all)
    lengths = pd.concat(meta, ignore_index=True)["length"].to_numpy()
    np.savez(PREFIX_FILE, header=kitsune_lib.prefix_hashes(H_header, lengths), all_inputs=kitsune_lib.prefix_hashes(H_all, lengths))
    pd.DataFrame(iat_rows).set_index("capture").to_csv(kitsune_lib.RESULT_DIR / "iat_resolution.csv")

X = np.load(SEQ_FILE, mmap_mode="r")  # segments x L x columns, zero-padded after the last packet
segments = pd.read_parquet(SEG_FILE)
print(f"\n{X.shape[0]:,} segments, {X.shape[2]} columns, {X.nbytes / 1e9:.1f} GB")
segments.groupby("capture", sort=False).agg(segments=("attack", "size"), attack=("attack", "sum"), median_length=("length", "median"))

In [ ]:
# What a sample looks like: a few segments, then the model inputs of one of them, packet by packet
print("Segments (one row per sample):")
print(segments.groupby("capture", sort=False).sample(1, random_state=SEED).to_string(index=False))

example = segments[(segments["capture"] == "SSL Renegotiation") & (segments["attack"] == 1) & (segments["length"] >= 8)].iloc[0]
values = pd.DataFrame(np.array(X[int(example["segment"]), :8]), columns=COLUMNS).round(2)
print(f"\nSegment {int(example['segment']):,}: {example['capture']}, attack={int(example['attack'])}, {int(example['length'])} packets. "
      f"First 8 packets, before standardizing (frame_len, tcp_window and iat are log-scaled):")
print(values[["frame_len", "tcp_window", "ip_hlen", "l4_hlen", "iat", "direction", "tcp", "udp", "syn", "ack", "psh", "fin", "rst", "mask"]].to_string())

short = segments[segments["length"] == 2].iloc[0]
print(f"\nPadding: segment {int(short['segment']):,} has 2 packets, so `mask` is",
      np.array(X[int(short["segment"]), :4, COL["mask"]]).astype(int).tolist(), "for its first 4 steps")

## Stage 1: data, diagnostics and tracks

Reads the parsed segments built above (PCAP parsing, flows, 50-packet segments, inputs), measures the pre-split
diagnostics, and builds the two tracks. Every later stage reads the tracks saved here.

- **Diagnostics** (informational, before any split): the distinct share and the conflicting sequences.
- **Tracks**, each at its own full size and split by time (per capture and class: earliest 60% train, next 20%
  validation, last 20% test):

| Track | Duplicates | Conflicts | Scenario |
|---|---|---|---|
| A | kept | kept | labels as recorded |
| B | kept | dropped | best case after conflicts are reviewed by hand (not every conflict may be resolvable) |

Real-world results are expected to sit between A and B.

**Which key defines a conflict for Track B: "headers".** Two segments conflict when their packets agree on every header
field except TTL and destination port, and also ignoring the inter-arrival time (IAT), yet carry different labels.
Those three fields are left out on purpose: their values come from the testbed, not from the attack (TTL: the sending
host's operating system and hop count; destination port: the services this testbed runs; IAT: this network's link
speed, load and capture host). A conflict that only they could resolve would be resolved by the testbed's identity, not
by what the attack does, which is the dependence this project measures with the model without them. The rule is also
the stricter one: it removes every conflict either model could face, and one Track B shared by both models keeps them
on the same rows, so their difference comes from their inputs alone. Its cost is measured below: how many dropped
segments TTL, destination port or IAT would have separated. A Track B judged per model (on each model's own key) is
left for future work.

Mirai (another network) and ARP MitM (same network) are held out of every track.

In [ ]:
import numpy as np
import pandas as pd

from kitsune_lib import CAPTURES, KEYS, OOD_CAPTURES, L, DATA_DIR, TRACK_DIR, RESULT_DIR, load_data

X, segments, y_all, lengths, prefix_header, prefix_all = load_data()

# Flow of every segment, numbered as when the segments were built (captures in order, flows sorted by start time);
# used to resample whole flows for confidence intervals, since segments of one flow are not independent
from kitsune_lib import L as _L
flow_rows, offset = [], 0
for capture, folder in CAPTURES.items():
    p = pd.read_parquet(DATA_DIR / "packets" / f"{folder}.parquet", columns=["flow", "ts"]).sort_values(["flow", "ts"], kind="stable")
    first = (p.groupby("flow").cumcount().to_numpy() % _L) == 0
    flow_rows.append(pd.DataFrame({"segment": np.arange(offset, offset + first.sum()), "flow": p["flow"].to_numpy()[first]}))
    offset += int(first.sum())
flows = pd.concat(flow_rows, ignore_index=True)
assert offset == len(segments)
segments = segments.merge(flows, on="segment")
print(f"{len(segments):,} segments in {segments.groupby(['capture', 'flow']).ngroups:,} flows, {X.shape[2]} stored columns")

### Diagnostics before the split

Informational only. A sample at k is a segment cut to its first k packets. The **distinct share** counts one copy of each
distinct sequence, labels ignored, over all samples: below 1, copies can fall on both sides of a split, so leakage shows
up after it as the difference between the "all" and "unseen" views. **Conflicting sequences** carry both labels; they are
candidates for manual relabelling where the true label can be established, and the rest are genuinely ambiguous.
Two sequence keys (kitsune_lib.packet_keys), each exactly the inputs of one model, labels never included: "headers"
(the model without TTL, destination port and IAT) and "all inputs" (headers + TTL + destination port + IAT, the
main model).

In [ ]:
def diagnostics(hashes, y):
    c = pd.DataFrame({"h": hashes, "attack": y}).groupby("h")["attack"].agg(n="size", n_attack="sum")
    conflict = (c["n_attack"] > 0) & (c["n_attack"] < c["n"])
    return {"distinct share": len(c) / len(y), "conflicting sequences": int(conflict.sum()),
            "conflicting share of distinct": conflict.mean(), "samples in conflict": int(c.loc[conflict, "n"].sum())}


known = ~segments["capture"].isin(OOD_CAPTURES).to_numpy()
report = pd.concat({
    key: pd.DataFrame([diagnostics(H[known, k - 1], y_all[known]) for k in range(1, L + 1)], index=range(1, L + 1))
    for key, H in [("headers", prefix_header), ("all inputs", prefix_all)]
}, names=["key", "k"])
report.to_csv(RESULT_DIR / "diagnostics.csv")
report.loc[(["headers", "all inputs"], [1, 2, 3, 5, 10, 20, 50]), :].round(4)

In [ ]:
import matplotlib.pyplot as plt

FIG_DIR = DATA_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)
# Reference palette (dataviz skill): series in fixed order, recessive chrome
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK_2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "axes.titlecolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "legend.frameon": False, "font.size": 10,
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "DejaVu Sans"],
})

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for key, color in [("headers", BLUE), ("all inputs", ORANGE)]:
    g = report.loc[key]
    axes[0].plot(g.index, g["distinct share"], color=color, label=key)
    axes[1].plot(g.index, g["conflicting share of distinct"] * 100, color=color, label=key)
axes[0].set_title("Distinct share of k-packet prefixes", loc="left", fontsize=11)
axes[1].set_title("Conflicting sequences, % of distinct", loc="left", fontsize=11)
for ax in axes:
    ax.set_xlabel("Packets seen, k")
    ax.legend(loc="center right")
fig.tight_layout()
fig.savefig(FIG_DIR / "data_diagnostics.png", dpi=150)
plt.show()

In [ ]:
# Every conflicting whole segment for manual review, most lopsided first (a 10-to-1 split hints at a mislabel;
# a balanced one at genuine ambiguity). The ratio only orders the list, it decides nothing.
in_dist = segments[known].copy()
in_dist["capture_order"] = in_dist["capture"].map({c: i for i, c in enumerate(CAPTURES)})
in_dist = in_dist.sort_values(["capture_order", "start"], kind="stable")
copies = in_dist.groupby("hash").agg(copies=("segment", "size"), captures=("capture", "nunique"),
                                     benign=("attack", lambda a: int((a == 0).sum())), attack=("attack", "sum"),
                                     length=("length", "first"))
review = copies[(copies["benign"] > 0) & (copies["attack"] > 0)]
review = review.assign(minority_share=review[["benign", "attack"]].min(axis=1) / review["copies"]).sort_values("minority_share")
review.to_csv(RESULT_DIR / "conflicts_for_review.csv")
print(f"{len(review):,} conflicting sequences ({int(review['copies'].sum()):,} segments) written for review")
review.head(10)

### Tracks

Built from the known captures in time order. Each track keeps its own size: nothing is sampled to match the other.
The split is by time within each capture and class.

In [ ]:
conflict_hashes = review.index
in_conflict = in_dist["hash"].isin(conflict_hashes)
tracks = {"A": in_dist, "B": in_dist[~in_conflict]}

# The cost of judging conflicts on "headers": dropped segments that TTL, destination port or IAT would have separated
# (not in conflict under "all inputs")
by_all = in_dist.groupby("hash_all")["attack"].agg(["min", "max"])
in_conflict_all = in_dist["hash_all"].isin(by_all.index[by_all["min"] != by_all["max"]])
b_drops = pd.DataFrame([{"segments dropped from Track B": int(in_conflict.sum()),
                         "of which also in conflict under all inputs": int((in_conflict & in_conflict_all).sum()),
                         "of which TTL, destination port or IAT would separate": int((in_conflict & ~in_conflict_all).sum()),
                         "known segments": len(in_dist)}])
b_drops.to_csv(RESULT_DIR / "track_b_drops.csv", index=False)
display(b_drops.T)


def time_split(g):
    rank = (np.arange(len(g)) + 0.5) / len(g)
    return pd.Series(np.select([rank < 0.6, rank < 0.8], ["train", "val"], "test"), index=g.index)


summary = []
for name, t in tracks.items():
    t = t.sort_values(["capture_order", "attack", "start"], kind="stable").copy()
    t["split"] = pd.concat([time_split(g) for _, g in t.groupby(["capture", "attack"], sort=False)])
    for _, g in t.groupby(["capture", "attack"]):  # time order holds inside each capture and class
        s = g.groupby("split")["start"]
        assert s.max()["train"] <= s.min()["val"] and s.max()["val"] <= s.min()["test"]
    t[["segment", "capture", "flow", "attack", "length", "start", "hash", "split"]].to_parquet(TRACK_DIR / f"{name}.parquet", index=False)
    tracks[name] = t
    train_h = {key: set(t.loc[t["split"] == "train", col]) for key, col in KEYS.items()}
    for part in ["train", "val", "test"]:
        d = t[t["split"] == part]
        summary.append({"track": name, "split": part, "segments": len(d), "attack share": d["attack"].mean(),
                        **{("unseen rows" if key == "headers" else f"unseen rows, {key}"):
                           int((~d[col].isin(train_h[key])).sum()) if part != "train" else np.nan for key, col in KEYS.items()}})
assert not tracks["B"]["hash"].isin(conflict_hashes).any()
summary = pd.DataFrame(summary).set_index(["track", "split"])
summary.to_csv(RESULT_DIR / "track_sizes.csv")
summary

In [ ]:
# Held out of every track: attack segments of Mirai (another network) and ARP MitM (same network) whose sequence never
# occurs in the known captures, plus all held-out segments for the before/after views of Mirai's time-based labels
held = segments[~known].copy()
held["group"] = np.where(held["capture"] == "Mirai", "Mirai",
                         np.where(held["arp_share"] > 0.5, "ARP MitM (ARP)", "ARP MitM (relayed)"))
held["new sequence"] = ~held["hash"].isin(in_dist["hash"])
held[["segment", "capture", "flow", "attack", "length", "start", "hash", "group", "new sequence"]].to_parquet(TRACK_DIR / "held_out.parquet", index=False)
held.groupby(["group", "attack"]).agg(segments=("segment", "size"), new=("new sequence", "sum"))

## Stage 2: models (per track)

The track comes from the `TRACK` environment variable (A or B). Everything here uses only that track's data, and
every choice is made on its validation split, never on test.

1. **GRU size** (64, 128, 128x2) on all feature groups, by validation loss. The chosen size model is the **main model**.
2. **Model without TTL, destination port and IAT**: the same size on every other feature group. These inputs can
   identify this testbed rather than the attack (TTL: the sender's operating system and hop count; destination port:
   the attacked service; IAT: this network's link speed, load and capture host), so comparing the two models shows how
   much of the performance depends on them.

Losses are shown on all validation rows and on the unseen ones (sequence not in training, judged on the model's own key:
"all inputs" for models on every feature group, "headers" for the model without TTL, destination port and IAT).

### Track A

In [ ]:
%reset -f
import gc
import os

gc.collect()
os.environ["TRACK"] = "A"  # the stage reads its track from here

In [ ]:
import os

import numpy as np
import pandas as pd

from kitsune_lib import (GROUPS, GRU_GRID, KEYS, RESULT_DIR, SEED, TESTBED_GROUPS, eval_loss, fit_gru, group_columns,
                         load, load_data, model_key, read_track, rows_digest, save_json, split_index, fit_stats, unseen)

TRACK = os.environ.get("TRACK", "A")
SEEDS = [SEED]  # more seeds left for future work

X, segments, y_all, lengths, prefix_header, _ = load_data()
part = split_index(read_track(TRACK))
stats = fit_stats(X, part["train"])
x_train_all = load(X, part["train"], stats)
x_val_all = load(X, part["val"], stats)
y_train = y_all[part["train"]].astype(np.float32)
y_val = y_all[part["val"]].astype(np.float32)
# Unseen validation rows under each key; each model is read on its own key (kitsune_lib.model_key)
val_unseen = {key: unseen(part["val"], part["train"], segments[col].to_numpy()) for key, col in KEYS.items()}
weight = float((y_train == 0).sum() / (y_train == 1).sum())
print(f"Track {TRACK}: train {len(y_train):,}, validation {len(y_val):,}, unseen: "
      + ", ".join(f"{m.mean():.1%} ({key})" for key, m in val_unseen.items()))


def train(name, groups, arch, seed=SEED):
    cols = group_columns(groups)
    model, loss_all = fit_gru(f"{TRACK}_{name}", x_train_all[:, :, cols], y_train, x_val_all[:, :, cols], y_val,
                              *GRU_GRID[arch], seed=seed,
                              rows=rows_digest(TRACK, part["train"], part["val"], cols, stats, segments))
    m = val_unseen[model_key(groups)]
    loss_unseen = eval_loss(model, x_val_all[m][:, :, cols], y_val[m], weight, len(cols) - 1)
    return {"model": f"{TRACK}_{name}", "validation loss, all": loss_all, "validation loss, unseen": loss_unseen,
            "key": model_key(groups)}

#### 1. GRU size

In [ ]:
ALL = list(GROUPS)
size_rows = {arch: train(f"size_{arch}", ALL, arch) for arch in GRU_GRID}
sizes = pd.DataFrame(size_rows).T
arch = sizes["validation loss, all"].astype(float).idxmin()
sizes.to_csv(RESULT_DIR / f"{TRACK}_gru_size.csv")
print(f"chosen size: {arch}")
sizes

#### 2. Main model and model without TTL, destination port and IAT

In [ ]:
final = {"main": ALL, "without TTL, destination port and IAT":[g for g in ALL if g not in TESTBED_GROUPS]}
finals = []
for role, groups in final.items():
    tag = "all" if role == "main" else "no_testbed"
    for seed in SEEDS:
        if role == "main" and seed == SEED:
            r = {**size_rows[arch]}  # the chosen size model is the main model
        else:
            r = train(f"{arch}_{tag}" if seed == SEED else f"{arch}_{tag}_seed{seed}", groups, arch, seed)
        finals.append({"role": role, "seed": seed, "groups": ", ".join(groups), **r})
finals = pd.DataFrame(finals)
finals.to_csv(RESULT_DIR / f"{TRACK}_final_models.csv", index=False)
save_json({"track": TRACK, "arch": arch, "final": final,
           "models": {role: finals.loc[finals["role"] == role, "model"].tolist() for role in final}},
          RESULT_DIR / f"{TRACK}_selection.json")
finals

### Track B

In [ ]:
%reset -f
import gc
import os

gc.collect()
os.environ["TRACK"] = "B"  # the stage reads its track from here

In [ ]:
import os

import numpy as np
import pandas as pd

from kitsune_lib import (GROUPS, GRU_GRID, KEYS, RESULT_DIR, SEED, TESTBED_GROUPS, eval_loss, fit_gru, group_columns,
                         load, load_data, model_key, read_track, rows_digest, save_json, split_index, fit_stats, unseen)

TRACK = os.environ.get("TRACK", "A")
SEEDS = [SEED]  # more seeds left for future work

X, segments, y_all, lengths, prefix_header, _ = load_data()
part = split_index(read_track(TRACK))
stats = fit_stats(X, part["train"])
x_train_all = load(X, part["train"], stats)
x_val_all = load(X, part["val"], stats)
y_train = y_all[part["train"]].astype(np.float32)
y_val = y_all[part["val"]].astype(np.float32)
# Unseen validation rows under each key; each model is read on its own key (kitsune_lib.model_key)
val_unseen = {key: unseen(part["val"], part["train"], segments[col].to_numpy()) for key, col in KEYS.items()}
weight = float((y_train == 0).sum() / (y_train == 1).sum())
print(f"Track {TRACK}: train {len(y_train):,}, validation {len(y_val):,}, unseen: "
      + ", ".join(f"{m.mean():.1%} ({key})" for key, m in val_unseen.items()))


def train(name, groups, arch, seed=SEED):
    cols = group_columns(groups)
    model, loss_all = fit_gru(f"{TRACK}_{name}", x_train_all[:, :, cols], y_train, x_val_all[:, :, cols], y_val,
                              *GRU_GRID[arch], seed=seed,
                              rows=rows_digest(TRACK, part["train"], part["val"], cols, stats, segments))
    m = val_unseen[model_key(groups)]
    loss_unseen = eval_loss(model, x_val_all[m][:, :, cols], y_val[m], weight, len(cols) - 1)
    return {"model": f"{TRACK}_{name}", "validation loss, all": loss_all, "validation loss, unseen": loss_unseen,
            "key": model_key(groups)}

#### 1. GRU size

In [ ]:
ALL = list(GROUPS)
size_rows = {arch: train(f"size_{arch}", ALL, arch) for arch in GRU_GRID}
sizes = pd.DataFrame(size_rows).T
arch = sizes["validation loss, all"].astype(float).idxmin()
sizes.to_csv(RESULT_DIR / f"{TRACK}_gru_size.csv")
print(f"chosen size: {arch}")
sizes

#### 2. Main model and model without TTL, destination port and IAT

In [ ]:
final = {"main": ALL, "without TTL, destination port and IAT":[g for g in ALL if g not in TESTBED_GROUPS]}
finals = []
for role, groups in final.items():
    tag = "all" if role == "main" else "no_testbed"
    for seed in SEEDS:
        if role == "main" and seed == SEED:
            r = {**size_rows[arch]}  # the chosen size model is the main model
        else:
            r = train(f"{arch}_{tag}" if seed == SEED else f"{arch}_{tag}_seed{seed}", groups, arch, seed)
        finals.append({"role": role, "seed": seed, "groups": ", ".join(groups), **r})
finals = pd.DataFrame(finals)
finals.to_csv(RESULT_DIR / f"{TRACK}_final_models.csv", index=False)
save_json({"track": TRACK, "arch": arch, "final": final,
           "models": {role: finals.loc[finals["role"] == role, "model"].tolist() for role in final}},
          RESULT_DIR / f"{TRACK}_selection.json")
finals

## Stage 3: evaluation (per track)

The track comes from `TRACK`. Each final model from stage 2 becomes one **deployable unit**, built only from this track's
data: input statistics (training), temperatures (first validation half), two OOD detectors on training hidden states
(an Isolation Forest, and the Mahalanobis distance to the nearer class mean with a shared covariance), OOD limits and
the High/Low cut-offs (second validation half). Test and the held-out captures are used only to measure.

**Decision rule**, per segment, packet by packet up to K (K = L = 50 is the working cut-off; shorter K are reported):
OOD if the running mean of either detector's score is over its limit (forest OR Mahalanobis); High (block) if
P(attack) >= tau_attack; Low (benign, this segment only) at the segment's end if P(attack) <= tau_benign; otherwise
Medium ("I don't know").

**Thresholds** are per-track precision targets on the second validation half: tau_attack is the lowest cut-off with at
least 99% of blocks real attacks, tau_benign the highest with at least 99.9% of Low answers really benign. The two OOD
limits sit at the same quantile of their detector's known validation scores, the lowest at which at most 5% of known
validation segments are ever flagged by either detector.

**Views**: "all" is the split as it is; "unseen" is "all" with every row whose sequence appears in this track's training
split removed, judged on each model's own key ("all inputs" for the main model and the baselines, "headers" for the model
without TTL, destination port and IAT). Ceilings (MaxAcc: Jerabek et al. Eq. 1; minimum FPR: their lower bound) are computed on
exactly the rows each result is measured on.

### Track A

In [ ]:
%reset -f
import gc
import os

gc.collect()
os.environ["TRACK"] = "A"  # the stage reads its track from here

In [ ]:
import copy
import json
import os
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

import kitsune_lib as kl
from kitsune_lib import (BUCKETS, CAPTURES, GRU_GRID, K_REPORT, KEYS, L, SEED, STEP_BUCKET, at_prefix, at_rate, ceiling,
                         fit_stats, flow_bootstrap, group_columns, known_metrics, load, load_data, load_gru, load_json,
                         model_key, p_attack, predict, read_track, real_steps, route, rows_digest, scores, split_index,
                         tune_targets)

TRACK = os.environ.get("TRACK", "A")
OUT = kl.RESULT_DIR
HIDDEN_TRAIN, FOREST_STEPS = 30_000, 200_000
DEPLOY_RATE = 0.01  # attack share assumed for the "real network" precision line

X, segments, y_all, lengths, prefix_header, prefix_all = load_data()
track = read_track(TRACK).set_index("segment", drop=False)
part = split_index(track)
held = pd.read_parquet(kl.TRACK_DIR / "held_out.parquet").set_index("segment", drop=False)
selection = load_json(OUT / f"{TRACK}_selection.json")
arch = selection["arch"]
print(f"Track {TRACK}: size {arch}, main groups: {selection['final']['main']}")

# The two validation halves, stratified by capture and class
from sklearn.model_selection import train_test_split
v = track.loc[part["val"]]
strata = (v["capture"] + v["attack"].astype(str)).to_numpy()
counts = pd.Series(strata).map(pd.Series(strata).value_counts()).to_numpy()
strata = np.where(counts < 2, v["attack"].astype(str).to_numpy(), strata)  # a group of one: stratify by class only
cal_idx, thr_idx = train_test_split(part["val"], test_size=0.5, random_state=SEED,
                                    stratify=strata if pd.Series(strata).value_counts().min() >= 2 else None)
parts = {"train sample": np.sort(np.random.default_rng(SEED).choice(part["train"], min(HIDDEN_TRAIN, len(part["train"])), replace=False)),
         "cal": np.sort(cal_idx), "thr": np.sort(thr_idx), "val": part["val"], "test": part["test"], "held": held.index.to_numpy()}
# Unseen rows of every part under each key; each unit uses its model's key (u["unseen"])
unseen_by_key = {key: {p: ~np.isin(segments[col].to_numpy()[i], segments[col].to_numpy()[part["train"]]) for p, i in parts.items()}
                 for key, col in KEYS.items()}
for key, u_of in unseen_by_key.items():
    print(key, {p: f"{len(i):,} ({u_of[p].mean():.0%} unseen)" for p, i in parts.items()})

#### Deployable units

`build_unit` builds everything a deployment of one model needs and saves it. The main model uses all feature
groups; the model without TTL, destination port and IAT is built the same way.

In [ ]:
stats = fit_stats(X, part["train"])


def unit_for(model_name, groups):
    cols = group_columns(groups)
    model = load_gru(model_name, len(cols), *GRU_GRID[arch], rows=rows_digest(TRACK, part["train"], part["val"], cols, stats, segments))
    u = kl.build_unit(X, model, cols, stats, parts, lengths, y_all, forest_steps=FOREST_STEPS)
    u.update(model=model_name, groups=groups, key=model_key(groups), unseen=unseen_by_key[model_key(groups)],
             **{"model weights": kl.weights_digest(model.state_dict())})  # the unit is valid with these weights only
    joblib.dump({k: u[k] for k in ["model", "groups", "columns", "stats", "temperatures", "fitted temperatures", "detectors",
                                   "thresholds", "model weights"]},
                kl.MODEL_DIR / f"{model_name}_unit.joblib")
    return u


units = {}
for role, names in selection["models"].items():
    for name in names:
        t0 = time.time()
        units[name] = unit_for(name, selection["final"][role])
        print(f"{name}: unit built in {time.time() - t0:.0f}s")
main = units[selection["models"]["main"][0]]
thresholds = pd.DataFrame({(name, K): {k: v for k, v in t.items() if k != "ood limits"} for name, u in units.items()
                           for K, t in u["thresholds"].items()}).T
thresholds.index.names = ["model", "K"]
thresholds.to_csv(OUT / f"{TRACK}_thresholds.csv")
thresholds.xs(L, level="K")

#### Known traffic: pipeline results, "I know" performance, all and unseen

In [ ]:
def decide(u, p, K):
    t = u["tuned"][K]
    return route(u["prob"][p], t["flags"][p], u["n"][p], t["tau_attack"], t["tau_benign"], K)


rows = []
for name, u in units.items():
    for p in ("val", "test"):
        for K in K_REPORT:
            out, packets = decide(u, p, K)
            for view, m in [("all", np.ones(len(out), bool)), ("unseen", u["unseen"][p])]:
                r = known_metrics(out[m], u["y"][p][m], packets[m])
                r["block precision at 1% attacks"] = at_rate(r["attacks blocked"], r["benign blocked"], DEPLOY_RATE)
                rows.append({"model": name, "split": p, "K": K, "view": view, **r})
pipeline = pd.DataFrame(rows).set_index(["model", "split", "K", "view"])
pipeline.to_csv(OUT / f"{TRACK}_pipeline.csv")
shown = ["segments", "coverage", "attacks blocked", "block precision", "block precision at 1% attacks", "attacks called benign",
         "benign blocked", "benign called Low", "Low precision", "accuracy when answering", "macro F1 when answering",
         "Medium", "OOD", "mean packets to block"]
pipeline.loc[(main["model"], ["val", "test"], L), shown].round(4)

In [ ]:
# 95% intervals on test, resampling whole flows (segments of one flow are not independent)
flow_key = (track.loc[parts["test"], "capture"] + "/" + track.loc[parts["test"], "flow"].astype(str)).to_numpy()
out, _ = decide(main, "test", L)
cis = {view: flow_bootstrap(out[m], main["y"]["test"][m], flow_key[m]) for view, m in
       [("all", np.ones(len(out), bool)), ("unseen", main["unseen"]["test"])]}
pd.DataFrame(cis).to_csv(OUT / f"{TRACK}_bootstrap.csv")
pd.DataFrame(cis)

#### Per attack type, with the average across types

In [ ]:
capture_of = track.loc[parts["test"], "capture"].to_numpy()
start_of = track.loc[parts["test"], "start"].to_numpy()
out, packets = decide(main, "test", L)
rows = []
for view, vm in [("all", np.ones(len(out), bool)), ("unseen", main["unseen"]["test"])]:
    for capture in CAPTURES:
        m = vm & (capture_of == capture)
        if not m.any():
            continue
        r = known_metrics(out[m], main["y"]["test"][m], packets[m])
        hours = max((start_of[capture_of == capture].max() - start_of[capture_of == capture].min()) / 3600, 1e-9)
        y_m, o_m = main["y"]["test"][m], out[m]
        r.update({"test hours": hours, "false blocks per hour": ((o_m == "High") & (y_m == 0)).sum() / hours,
                  "Medium per hour": (o_m == "Medium").sum() / hours,
                  "attacks passed per hour": ((o_m != "High") & (y_m == 1)).sum() / hours})
        rows.append({"view": view, "attack type": capture, **r})
per_type = pd.DataFrame(rows).set_index(["view", "attack type"])
for view in ["all", "unseen"]:  # every attack type weighs the same in the average
    per_type.loc[(view, "average across types"), :] = per_type.loc[view].mean(numeric_only=True)
per_type.to_csv(OUT / f"{TRACK}_per_attack_type.csv")
per_type[["attack segments", "attacks blocked", "attacks called benign", "Medium", "block precision", "Low precision",
          "false blocks per hour", "attacks passed per hour"]].round(4)

In [ ]:
# Drift: four time slices of each capture's test period
rank = pd.Series(start_of).groupby(capture_of).rank(pct=True).to_numpy()
slice_of = np.minimum((rank * 4).astype(int), 3) + 1
drift = pd.DataFrame([{"slice": s, **known_metrics(out[slice_of == s], main["y"]["test"][slice_of == s])} for s in range(1, 5)]).set_index("slice")
drift.to_csv(OUT / f"{TRACK}_drift.csv")
drift[["segments", "coverage", "attacks blocked", "block precision", "Low precision", "Medium", "OOD"]].round(4)

#### Time to a decision

How much traffic time passes before an answer: the seconds between a segment's first packet and the packet its answer
came at (from the packets' inter-arrival times), for each decision packet K. "Blocks" are High answers on attack
segments; "any answer" covers every segment. Compute time is separate (deployment cost below).

In [ ]:
iat_s = np.expm1(np.array(X[parts["test"], :, kl.COL["iat"]], dtype=np.float64)) / 1e6  # seconds since the flow's previous packet
iat_s[:, 0] = 0  # counted from the segment's first packet
elapsed = np.cumsum(iat_s, axis=1)
rows = []
for name, u in units.items():
    y = u["y"]["test"]
    for K in K_REPORT:
        out, packets = decide(u, "test", K)
        t = elapsed[np.arange(len(out)), packets - 1]
        for answers, m in [("blocks", (out == "High") & (y == 1)), ("any answer", np.ones(len(out), bool))]:
            rows.append({"model": name, "K": K, "answers": answers, "segments": int(m.sum()),
                         "packets, mean": packets[m].mean() if m.any() else np.nan,
                         "seconds, median": np.median(t[m]) if m.any() else np.nan,
                         "seconds, 90th percentile": np.quantile(t[m], 0.9) if m.any() else np.nan})
decision_time = pd.DataFrame(rows).set_index(["model", "K", "answers"])
decision_time.to_csv(OUT / f"{TRACK}_decision_time.csv")
decision_time.round(4)

#### Forced answers and ceilings on the same rows

A ceiling bounds a model only when identical rows are identical *in what the model reads*. Each model has its own key:
"headers" holds exactly the inputs of the model without TTL, destination port and IAT, and "all inputs" those of the
main model (and of the baselines, which read the same inputs), so each model's ceilings on its own key are upper bounds
for it. Both keys are shown for every model. "above own ceiling" flags a forced accuracy above the MaxAcc of the model's
own key on the same rows, which should not happen: the only way it can is the IAT, held in whole microseconds by the
key (data/results/iat_resolution.csv counts the gaps that are not).

In [ ]:
PREFIXES = [1, 2, 3, 5, 10, 20, 50]
rows = []
for name, u in units.items():
    for p in ("val", "test"):
        idx = parts[p]
        for view, m in [("all", np.ones(len(idx), bool)), ("unseen", u["unseen"][p])]:
            for k in PREFIXES:
                pred = (at_prefix(u["prob"][p], u["n"][p], k)[m] >= 0.5).astype(int)
                r = {"model": name, "split": p, "view": view, "k": k, **scores(u["y"][p][m], pred)}
                for key, H in [("all inputs", prefix_all), ("headers", prefix_header)]:
                    c = ceiling(H[idx[m], k - 1], u["y"][p][m])
                    r.update({f"{cname}, {key}": value for cname, value in c.items() if cname != "rows"})
                r["own key"] = u["key"]
                r["above own ceiling"] = bool(r["accuracy"] > r[f"MaxAcc, {u['key']}"] + 1e-12)
                r["rows"] = int(m.sum())
                rows.append(r)
forced = pd.DataFrame(rows).set_index(["model", "split", "view", "k"])
forced.to_csv(OUT / f"{TRACK}_forced_and_ceilings.csv")
if forced["above own ceiling"].any():
    print(f"WARNING: {int(forced['above own ceiling'].sum())} rows of {TRACK}_forced_and_ceilings.csv score above their own "
          f"ceiling; see iat_resolution.csv", flush=True)
no_testbed = selection["models"]["without TTL, destination port and IAT"][0]
# The model without the testbed group, against its exact ceilings (key "headers")
forced.loc[(no_testbed, "test"), ["rows", "accuracy", "MaxAcc, headers", "macro F1", "MaxF1, headers", "FPR",
                                  "minimum FPR, headers", "missed attacks", "minimum missed attacks, headers"]].round(4)

In [ ]:
# The main model, against its exact ceilings (key "all inputs")
forced.loc[(main["model"], "test"), ["rows", "accuracy", "MaxAcc, all inputs", "macro F1", "MaxF1, all inputs", "FPR",
                                     "minimum FPR, all inputs", "missed attacks", "minimum missed attacks, all inputs"]].round(4)

#### Calibration on test

Temperature scaling divides the logit by one temperature per range of packets seen, fitted on the first validation half
and capped at T >= 1 so that calibration never makes the model more confident. Each model is scored with the
temperatures it uses (capped) and, for comparison, with the fitted ones (uncapped), at decision packets 1, 3, 10 and 50.
ECE: 15 equal-width bins of P(attack), the gap between mean P(attack) and the share of attacks in each bin, weighted by
the bin's size. The reliability bins at K = 50 are saved for the reliability diagram.

In [ ]:
def calibration(p, y, bins=15):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    which = np.minimum((p * bins).astype(int), bins - 1)
    ece = sum(abs(p[which == b].mean() - y[which == b].mean()) * (which == b).mean() for b in range(bins) if (which == b).any())
    return {"ECE": ece, "Brier": np.mean((p - y) ** 2), "NLL": -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))}


fitted_T = pd.DataFrame({name: dict(zip(BUCKETS, u["fitted temperatures"][[list(STEP_BUCKET).index(b) for b in range(len(BUCKETS))]]))
                         for name, u in units.items()}).T
print("fitted temperatures (before the cap), per range of packets seen:")
print(fitted_T.round(3))
rows, reliability = [], []
for name, u in units.items():
    for reading, T in [("capped (used)", u["temperatures"]), ("fitted, uncapped", u["fitted temperatures"])]:
        prob = p_attack(u["z"]["test"], T)
        for view, m in [("all", np.ones(len(u["y"]["test"]), bool)), ("unseen", u["unseen"]["test"])]:
            for k in [1, 3, 10, L]:
                p, y = at_prefix(prob, u["n"]["test"], k)[m], u["y"]["test"][m]
                rows.append({"model": name, "temperature": reading, "view": view, "k": k, **calibration(p, y)})
                if k == L and view == "all":
                    which = np.minimum((p * 15).astype(int), 14)
                    reliability += [{"model": name, "temperature": reading, "bin": b, "segments": int((which == b).sum()),
                                     "mean P(attack)": p[which == b].mean(), "attack share": y[which == b].mean()}
                                    for b in range(15) if (which == b).any()]
calib = pd.DataFrame(rows).set_index(["model", "temperature", "view", "k"])
calib.to_csv(OUT / f"{TRACK}_calibration.csv")
pd.DataFrame(reliability).to_csv(OUT / f"{TRACK}_reliability.csv", index=False)
calib.xs("all", level="view").round(4)

Does the cap change the answers? With the uncapped temperatures, the High/Low cut-offs are retuned on the second
validation half (same precision targets, same OOD flags) and the test answers are compared with the ones in use.

In [ ]:
rows = []
for name, u in units.items():
    prob = {p: p_attack(u["z"][p], u["fitted temperatures"]) for p in ("thr", "test")}
    for K in [1, 3, 10, L]:
        flags = u["tuned"][K]["flags"]
        t = tune_targets(prob["thr"], flags["thr"], u["n"]["thr"], u["y"]["thr"], K)
        uncapped = route(prob["test"], flags["test"], u["n"]["test"], t["tau_attack"], t["tau_benign"], K)
        for reading, (out, packets) in [("capped (used)", decide(u, "test", K)), ("fitted, uncapped", uncapped)]:
            for view, m in [("all", np.ones(len(out), bool)), ("unseen", u["unseen"]["test"])]:
                rows.append({"model": name, "temperature": reading, "K": K, "view": view,
                             **known_metrics(out[m], u["y"]["test"][m], packets[m])})
cap_answers = pd.DataFrame(rows).set_index(["model", "temperature", "K", "view"])
cap_answers.to_csv(OUT / f"{TRACK}_cap_answers.csv")
cap_answers.xs("all", level="view")[["coverage", "attacks blocked", "block precision", "attacks called benign", "Low precision",
                                     "Medium", "mean packets to block"]].round(4)

#### Held-out captures: Mirai (another network) and ARP MitM (same network)

On held-out traffic the only correct answer is "I don't know" (Medium or OOD); any High or Low answer is a failure, even
when it matches the label. Mirai's labels are time-based (every packet from 121,622 on counts as attack), so its groups
are read as before and during infection.

In [ ]:
group_of = held.loc[parts["held"], "group"].to_numpy()
cap_of = held.loc[parts["held"], "capture"].to_numpy()
label_of = main["y"]["held"]
rows = []
for name, u in units.items():
    out, _ = decide(u, "held", L)
    for view, vm in [("all", np.ones(len(out), bool)), ("unseen", u["unseen"]["held"])]:
        for capture in ["Mirai", "ARP MitM"]:
            for lab, desc in [(0, "before infection" if capture == "Mirai" else "benign"), (1, "during infection" if capture == "Mirai" else "attack")]:
                m = vm & (cap_of == capture) & (label_of == lab)
                o = out[m]
                rows.append({"model": name, "view": view, "traffic": f"{capture}, {desc}", "segments": int(m.sum()),
                             "failure rate": np.isin(o, ["High", "Low"]).mean(), **{a: (o == a).mean() for a in ["High", "Low", "Medium", "OOD"]}})
held_table = pd.DataFrame(rows).set_index(["model", "view", "traffic"])
held_table.to_csv(OUT / f"{TRACK}_held_out.csv")
held_table.loc[main["model"]].round(4)

In [ ]:
# OOD separation: AUROC of each detector's running mean, known test packets against held-out packets, per prefix bucket
real_t, real_h = real_steps(main["n"]["test"]), real_steps(main["n"]["held"])
rows = []
for detector in kl.DETECTORS:
    s = {p: main["mean"][p][detector] for p in ("test", "held")}
    for group in pd.unique(group_of):
        for b, bucket in enumerate(BUCKETS):
            inb = (STEP_BUCKET == b)[None, :]
            known_s = s["test"][real_t & inb]
            new_s = s["held"][(group_of == group)[:, None] & real_h & inb]
            if len(new_s):
                rows.append({"detector": detector, "held-out": group, "bucket": bucket,
                             "AUROC": roc_auc_score(np.r_[np.zeros(len(known_s)), np.ones(len(new_s))], np.r_[known_s, new_s])})
ood_auc = pd.DataFrame(rows).set_index(["detector", "held-out", "bucket"])["AUROC"].unstack("bucket")[list(BUCKETS)]
ood_auc.to_csv(OUT / f"{TRACK}_ood_auroc.csv")
ood_auc.round(3)

Which detector flags what, at K = 50: the share of segments flagged by the forest only, Mahalanobis only, both, and
either (the OOD answer). Known test traffic shows the cost, the held-out captures what each detector adds.

In [ ]:
rows = []
for name, u in units.items():
    t = u["tuned"][L]
    for p, groups in [("test", np.where(u["y"]["test"] == 1, "known test, attack", "known test, benign")), ("held", group_of)]:
        live = real_steps(u["n"][p])
        by = {d: ((u["mean"][p][d] > np.asarray(t["ood limits"][d])[STEP_BUCKET][None, :]) & live).any(axis=1) for d in kl.DETECTORS}
        for g in pd.unique(groups):
            m = groups == g
            f, mh = by["forest"][m], by["mahalanobis"][m]
            rows.append({"model": name, "traffic": g, "segments": int(m.sum()), "forest only": (f & ~mh).mean(),
                         "Mahalanobis only": (mh & ~f).mean(), "both": (f & mh).mean(), "either": (f | mh).mean()})
detector_overlap = pd.DataFrame(rows).set_index(["model", "traffic"])
detector_overlap.to_csv(OUT / f"{TRACK}_ood_detectors.csv")
detector_overlap.round(4)

#### Baselines: k-NN and XGBoost against the GRU

One section, one training per model: k-NN (L1 distance, 15 neighbours, share of attack among them) and weighted
XGBoost, at each checkpoint k on the first k packets of the main model's inputs. Every model is read at the
same checkpoints with the OOD gate off, gets its High/Low cut-offs from the same precision targets on the same rows of
the second validation half, and is scored on the same 15,000 test rows. Forced answers (P(attack) >= 0.5) sit next to the
ceilings on those rows; the H/M/L reading is the main comparison. These models read the main model's inputs, so the
ceilings are on the "all inputs" key and are upper bounds for every one of them ("above own ceiling" flags a breach).

In [ ]:
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBClassifier

CHECKS, EVAL_ROWS, KNN_TRAIN, NEIGHBOURS = [1, 2, 3, 5, 10, 20, L], 15_000, 40_000, 15
rng = np.random.default_rng(SEED)
pick = {p: np.sort(rng.choice(len(parts[p]), min(EVAL_ROWS, len(parts[p])), replace=False)) for p in ("thr", "test")}
cols = main["columns"]
cache = OUT / f"{TRACK}_baseline_probs.npz"
# What the cached probabilities were computed from: the settings, the rows (training, evaluated), the inputs read and
# their standardisation, and the labels
baseline_fingerprint = {"training version": kl.TRAINING_VERSION, "checks": CHECKS, "evaluated rows": EVAL_ROWS, "k-NN training rows": KNN_TRAIN, "neighbours": NEIGHBOURS,
                        "xgboost": "n_estimators=200, max_depth=6, learning_rate=0.1, hist", "seed": SEED,
                        "data": kl.digest(part["train"], parts["thr"][pick["thr"]], parts["test"][pick["test"]], np.asarray(cols),
                                          stats[0], stats[1], y_all[part["train"]], segments["hash_all"].to_numpy()[part["train"]])}
if cache.exists():
    probs = dict(np.load(cache))
    stored = probs.pop("fingerprint", None)
    kl.check_fingerprint(None if stored is None else json.loads(str(stored)), baseline_fingerprint, cache)
else:
    x_tr, y_tr = load(X, part["train"], stats, cols), y_all[part["train"]]
    knn_rows = rng.permutation(len(x_tr))[:KNN_TRAIN]
    x_ev = {p: load(X, parts[p][pick[p]], stats, cols) for p in pick}
    probs = {f"{m}|{p}": np.zeros((len(pick[p]), len(CHECKS))) for m in ["k-NN", "XGBoost"] for p in pick}
    for j, k in enumerate(CHECKS):
        t0 = time.time()
        flat = x_tr[:, :k].reshape(len(x_tr), -1)
        nn = NearestNeighbors(n_neighbors=NEIGHBOURS, metric="manhattan", algorithm="brute", n_jobs=-1).fit(flat[knn_rows])
        xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, tree_method="hist", n_jobs=-1, random_state=SEED,
                            scale_pos_weight=(y_tr == 0).sum() / (y_tr == 1).sum()).fit(flat, y_tr)
        for p, x in x_ev.items():
            f = x[:, :k].reshape(len(x), -1)
            probs[f"k-NN|{p}"][:, j] = y_tr[knn_rows][nn.kneighbors(f, return_distance=False)].mean(axis=1)
            probs[f"XGBoost|{p}"][:, j] = xgb.predict_proba(f)[:, 1]
        print(f"k={k}: {time.time() - t0:.0f}s")
    np.savez(cache, fingerprint=np.array(json.dumps(baseline_fingerprint)), **probs)
for p in pick:  # the GRU read at the same checkpoints
    probs[f"GRU|{p}"] = np.stack([at_prefix(main["prob"][p][pick[p]], main["n"][p][pick[p]], k) for k in CHECKS], axis=1)

rows, forced_rows = [], []
y_ev = {p: main["y"][p][pick[p]] for p in pick}
full = {p: np.full(len(pick[p]), len(CHECKS)) for p in pick}
no_flags = {p: np.zeros((len(pick[p]), len(CHECKS)), bool) for p in pick}
test_idx = parts["test"][pick["test"]]
test_unseen = main["unseen"]["test"][pick["test"]]
for m in ["GRU", "k-NN", "XGBoost"]:
    t = tune_targets(probs[f"{m}|thr"], no_flags["thr"], full["thr"], y_ev["thr"], len(CHECKS))
    out, _ = route(probs[f"{m}|test"], no_flags["test"], full["test"], t["tau_attack"], t["tau_benign"], len(CHECKS))
    for view, vm in [("all", np.ones(len(out), bool)), ("unseen", test_unseen)]:
        rows.append({"model": m, "view": view, "tau_attack": t["tau_attack"], "tau_benign": t["tau_benign"],
                     **known_metrics(out[vm], y_ev["test"][vm])})
        for j, k in enumerate(CHECKS):
            r = {"model": m, "view": view, "k": k, **scores(y_ev["test"][vm], (probs[f"{m}|test"][vm, j] >= 0.5).astype(int))}
            c = ceiling(prefix_all[test_idx[vm], k - 1], y_ev["test"][vm])  # key "all inputs": exact for every model here
            forced_rows.append({**r, **{f"{name} (same rows)": value for name, value in c.items()},
                                "above own ceiling": bool(r["accuracy"] > c["MaxAcc"] + 1e-12)})
baselines = pd.DataFrame(rows).set_index(["model", "view"])
baselines_forced = pd.DataFrame(forced_rows).set_index(["model", "view", "k"])
baselines.to_csv(OUT / f"{TRACK}_baselines.csv")
baselines_forced.to_csv(OUT / f"{TRACK}_baselines_forced.csv")
if baselines_forced["above own ceiling"].any():
    print(f"WARNING: {int(baselines_forced['above own ceiling'].sum())} baseline rows score above their ceiling; "
          f"see iat_resolution.csv", flush=True)
baselines[["coverage", "attacks blocked", "block precision", "attacks called benign", "benign called Low", "Low precision",
           "accuracy when answering", "macro F1 when answering", "Medium"]].round(4)

In [ ]:
baselines_forced[["accuracy", "MaxAcc (same rows)", "macro F1", "MaxF1 (same rows)", "FPR", "missed attacks"]].round(4)

#### Main model vs without TTL, destination port and IAT

In [ ]:
roles = {n: role for role, names in selection["models"].items() for n in names}
compare = pd.concat({view: pipeline.xs(("test", L, view), level=["split", "K", "view"])[shown] for view in ["all", "unseen"]},
                    names=["view"]).rename(index=roles, level=1)
compare.to_csv(OUT / f"{TRACK}_testbed_comparison.csv")
compare.round(4)

#### Evasion: perturbing what an attacker controls

Test attack segments with timing jitter (each inter-arrival time multiplied by a factor in [0.8, 1.2]) or padding (up to
32 bytes added to each frame), with no retraining. A block that turns into Medium or OOD fails safe; one that turns into
Low is the dangerous failure.

In [ ]:
rng = np.random.default_rng(SEED)
att = parts["test"][main["y"]["test"] == 1]
raw = np.array(X[att])
mask = raw[:, :, kl.MASK_IDX] > 0
variants = {"none": raw}
v = raw.copy(); iat = np.expm1(v[:, :, kl.COL["iat"]]) * rng.uniform(0.8, 1.2, mask.shape); v[:, :, kl.COL["iat"]] = np.where(mask, np.log1p(iat), 0); variants["IAT jitter +/-20%"] = v
v = raw.copy(); size = np.expm1(v[:, :, kl.COL["frame_len"]]) + rng.integers(0, 33, mask.shape); v[:, :, kl.COL["frame_len"]] = np.where(mask, np.log1p(size), 0); variants["padding up to 32 bytes"] = v
mu, sd = main["stats"]
n_att = lengths[att]
t = main["tuned"][L]
rows = []
for vname, raw_v in variants.items():
    x = raw_v.copy()
    x[:, :, kl.CONT_IDX] = (x[:, :, kl.CONT_IDX] - mu) / sd
    x *= x[:, :, [kl.MASK_IDX]]
    z, h = predict(main["net"], x[:, :, main["columns"]].copy(), with_hidden=True)
    flags = kl.ood_flags(kl.ood_running_means(main["detectors"], h, n_att), t["ood limits"])
    del h
    out, _ = route(p_attack(z, main["temperatures"]), flags, n_att, t["tau_attack"], t["tau_benign"], L)
    rows.append({"perturbation": vname, **{a: (out == a).mean() for a in ["High", "Low", "Medium", "OOD"]}})
evasion = pd.DataFrame(rows).set_index("perturbation")
evasion.to_csv(OUT / f"{TRACK}_evasion.csv")
print(f"{len(att):,} test attack segments; inputs used: {main['groups']}")
evasion.round(4)

#### Deployment cost

Test segments of the main model are classified packet by packet, as a deployment would: the GRU step, both OOD scores
(Isolation Forest, Mahalanobis), their running means against the OOD limits, the temperature and the High/Low cut-offs,
stopping at the answer. Everything runs on one thread. The time per packet covers that whole path, with the share of
each part; the answers are checked against the batch decision rule used in evaluation, so the timed code is the logic
that was measured.

In [ ]:
COST_SEGMENTS = 300  # test segments timed
hidden, layers = GRU_GRID[arch]
net = main["net"].eval()
t_L = main["tuned"][L]
limits = {d: np.asarray(t_L["ood limits"][d]) for d in kl.DETECTORS}
forest = copy.deepcopy(main["detectors"]["forest"]).set_params(n_jobs=1)  # one thread, like the rest
maha = main["detectors"]["mahalanobis"]
PARTS = ["GRU", "Isolation Forest", "Mahalanobis", "decision rule"]


@torch.no_grad()
def classify_stream(x, n, K=L):
    """One segment, one packet at a time. Returns its answer, the packet it came at and the seconds spent in each part."""
    spent, sums, state = dict.fromkeys(PARTS, 0.0), np.zeros(len(kl.DETECTORS)), None
    last = min(int(n), K)
    for k in range(1, last + 1):
        t0 = time.perf_counter()
        o, state = net.gru(x[None, k - 1:k], state)
        z = float(net.head(o))
        h = o[0].numpy()
        t1 = time.perf_counter()
        score = {"forest": -forest.score_samples(h)[0]}
        t2 = time.perf_counter()
        score["mahalanobis"] = min(float((((h - mu) @ maha["precision"]) * (h - mu)).sum()) for mu in maha["means"])
        t3 = time.perf_counter()
        sums += [score[d] for d in kl.DETECTORS]
        flagged = any(sums[i] / k > limits[d][STEP_BUCKET[k - 1]] for i, d in enumerate(kl.DETECTORS))
        p = 1 / (1 + np.exp(z / main["temperatures"][k - 1]))
        answer = "OOD" if flagged else "High" if p >= t_L["tau_attack"] else None
        if answer is None and k == last:
            answer = "Low" if p <= t_L["tau_benign"] else "Medium"
        t4 = time.perf_counter()
        for part_name, s in zip(PARTS, [t1 - t0, t2 - t1, t3 - t2, t4 - t3]):
            spent[part_name] += s
        if answer:
            return answer, k, spent


torch.set_num_threads(1)
timed = np.sort(np.random.default_rng(SEED).choice(len(parts["test"]), min(COST_SEGMENTS, len(parts["test"])), replace=False))
x_timed = torch.from_numpy(load(X, parts["test"][timed], stats, main["columns"]))
n_timed = main["n"]["test"][timed]
classify_stream(x_timed[0], n_timed[0])  # warm-up, not counted
runs = [classify_stream(x_timed[i], n_timed[i]) for i in range(len(timed))]
torch.set_num_threads(os.cpu_count())
batch_out, batch_packets = decide(main, "test", L)
same = np.mean([(a, k) == (batch_out[j], batch_packets[j]) for (a, k, _), j in zip(runs, timed)])
if same < 0.99:  # the timed path must be the evaluated decision rule
    raise SystemExit(f"streaming answers agree with evaluation on only {same:.1%} of the timed segments")
timed_packets = sum(k for _, k, _ in runs)
spent = {part_name: sum(s[part_name] for _, _, s in runs) for part_name in PARTS}
per_packet_us = sum(spent.values()) / timed_packets * 1e6
# What a streaming deployment keeps per open flow between packets (float32 = 4 bytes)
state_parts = {
    "GRU hidden state (bytes)": layers * hidden * 4,          # one hidden vector per GRU layer
    "OOD running sums (bytes)": len(kl.DETECTORS) * 4,        # one running sum per detector; divided by the count below
    "packet counter (bytes)": 1,                              # packets seen in the current segment, at most L = 50
}
cost = {"inference per packet, one thread (us)": per_packet_us, "packets per second, one thread": 1e6 / per_packet_us,
        **{f"{part_name}, per packet (us)": s / timed_packets * 1e6 for part_name, s in spent.items()},
        "timed segments": len(timed), "timed packets": int(timed_packets), "answers equal to evaluation": float(same),
        **state_parts, "state per flow (bytes)": sum(state_parts.values()),
        "forest trees": main["detectors"]["forest"].n_estimators}
kl.save_json(cost, OUT / f"{TRACK}_deployment_cost.json")
cost

### Track B

In [ ]:
%reset -f
import gc
import os

gc.collect()
os.environ["TRACK"] = "B"  # the stage reads its track from here

In [ ]:
import copy
import json
import os
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

import kitsune_lib as kl
from kitsune_lib import (BUCKETS, CAPTURES, GRU_GRID, K_REPORT, KEYS, L, SEED, STEP_BUCKET, at_prefix, at_rate, ceiling,
                         fit_stats, flow_bootstrap, group_columns, known_metrics, load, load_data, load_gru, load_json,
                         model_key, p_attack, predict, read_track, real_steps, route, rows_digest, scores, split_index,
                         tune_targets)

TRACK = os.environ.get("TRACK", "A")
OUT = kl.RESULT_DIR
HIDDEN_TRAIN, FOREST_STEPS = 30_000, 200_000
DEPLOY_RATE = 0.01  # attack share assumed for the "real network" precision line

X, segments, y_all, lengths, prefix_header, prefix_all = load_data()
track = read_track(TRACK).set_index("segment", drop=False)
part = split_index(track)
held = pd.read_parquet(kl.TRACK_DIR / "held_out.parquet").set_index("segment", drop=False)
selection = load_json(OUT / f"{TRACK}_selection.json")
arch = selection["arch"]
print(f"Track {TRACK}: size {arch}, main groups: {selection['final']['main']}")

# The two validation halves, stratified by capture and class
from sklearn.model_selection import train_test_split
v = track.loc[part["val"]]
strata = (v["capture"] + v["attack"].astype(str)).to_numpy()
counts = pd.Series(strata).map(pd.Series(strata).value_counts()).to_numpy()
strata = np.where(counts < 2, v["attack"].astype(str).to_numpy(), strata)  # a group of one: stratify by class only
cal_idx, thr_idx = train_test_split(part["val"], test_size=0.5, random_state=SEED,
                                    stratify=strata if pd.Series(strata).value_counts().min() >= 2 else None)
parts = {"train sample": np.sort(np.random.default_rng(SEED).choice(part["train"], min(HIDDEN_TRAIN, len(part["train"])), replace=False)),
         "cal": np.sort(cal_idx), "thr": np.sort(thr_idx), "val": part["val"], "test": part["test"], "held": held.index.to_numpy()}
# Unseen rows of every part under each key; each unit uses its model's key (u["unseen"])
unseen_by_key = {key: {p: ~np.isin(segments[col].to_numpy()[i], segments[col].to_numpy()[part["train"]]) for p, i in parts.items()}
                 for key, col in KEYS.items()}
for key, u_of in unseen_by_key.items():
    print(key, {p: f"{len(i):,} ({u_of[p].mean():.0%} unseen)" for p, i in parts.items()})

#### Deployable units

`build_unit` builds everything a deployment of one model needs and saves it. The main model uses all feature
groups; the model without TTL, destination port and IAT is built the same way.

In [ ]:
stats = fit_stats(X, part["train"])


def unit_for(model_name, groups):
    cols = group_columns(groups)
    model = load_gru(model_name, len(cols), *GRU_GRID[arch], rows=rows_digest(TRACK, part["train"], part["val"], cols, stats, segments))
    u = kl.build_unit(X, model, cols, stats, parts, lengths, y_all, forest_steps=FOREST_STEPS)
    u.update(model=model_name, groups=groups, key=model_key(groups), unseen=unseen_by_key[model_key(groups)],
             **{"model weights": kl.weights_digest(model.state_dict())})  # the unit is valid with these weights only
    joblib.dump({k: u[k] for k in ["model", "groups", "columns", "stats", "temperatures", "fitted temperatures", "detectors",
                                   "thresholds", "model weights"]},
                kl.MODEL_DIR / f"{model_name}_unit.joblib")
    return u


units = {}
for role, names in selection["models"].items():
    for name in names:
        t0 = time.time()
        units[name] = unit_for(name, selection["final"][role])
        print(f"{name}: unit built in {time.time() - t0:.0f}s")
main = units[selection["models"]["main"][0]]
thresholds = pd.DataFrame({(name, K): {k: v for k, v in t.items() if k != "ood limits"} for name, u in units.items()
                           for K, t in u["thresholds"].items()}).T
thresholds.index.names = ["model", "K"]
thresholds.to_csv(OUT / f"{TRACK}_thresholds.csv")
thresholds.xs(L, level="K")

#### Known traffic: pipeline results, "I know" performance, all and unseen

In [ ]:
def decide(u, p, K):
    t = u["tuned"][K]
    return route(u["prob"][p], t["flags"][p], u["n"][p], t["tau_attack"], t["tau_benign"], K)


rows = []
for name, u in units.items():
    for p in ("val", "test"):
        for K in K_REPORT:
            out, packets = decide(u, p, K)
            for view, m in [("all", np.ones(len(out), bool)), ("unseen", u["unseen"][p])]:
                r = known_metrics(out[m], u["y"][p][m], packets[m])
                r["block precision at 1% attacks"] = at_rate(r["attacks blocked"], r["benign blocked"], DEPLOY_RATE)
                rows.append({"model": name, "split": p, "K": K, "view": view, **r})
pipeline = pd.DataFrame(rows).set_index(["model", "split", "K", "view"])
pipeline.to_csv(OUT / f"{TRACK}_pipeline.csv")
shown = ["segments", "coverage", "attacks blocked", "block precision", "block precision at 1% attacks", "attacks called benign",
         "benign blocked", "benign called Low", "Low precision", "accuracy when answering", "macro F1 when answering",
         "Medium", "OOD", "mean packets to block"]
pipeline.loc[(main["model"], ["val", "test"], L), shown].round(4)

In [ ]:
# 95% intervals on test, resampling whole flows (segments of one flow are not independent)
flow_key = (track.loc[parts["test"], "capture"] + "/" + track.loc[parts["test"], "flow"].astype(str)).to_numpy()
out, _ = decide(main, "test", L)
cis = {view: flow_bootstrap(out[m], main["y"]["test"][m], flow_key[m]) for view, m in
       [("all", np.ones(len(out), bool)), ("unseen", main["unseen"]["test"])]}
pd.DataFrame(cis).to_csv(OUT / f"{TRACK}_bootstrap.csv")
pd.DataFrame(cis)

#### Per attack type, with the average across types

In [ ]:
capture_of = track.loc[parts["test"], "capture"].to_numpy()
start_of = track.loc[parts["test"], "start"].to_numpy()
out, packets = decide(main, "test", L)
rows = []
for view, vm in [("all", np.ones(len(out), bool)), ("unseen", main["unseen"]["test"])]:
    for capture in CAPTURES:
        m = vm & (capture_of == capture)
        if not m.any():
            continue
        r = known_metrics(out[m], main["y"]["test"][m], packets[m])
        hours = max((start_of[capture_of == capture].max() - start_of[capture_of == capture].min()) / 3600, 1e-9)
        y_m, o_m = main["y"]["test"][m], out[m]
        r.update({"test hours": hours, "false blocks per hour": ((o_m == "High") & (y_m == 0)).sum() / hours,
                  "Medium per hour": (o_m == "Medium").sum() / hours,
                  "attacks passed per hour": ((o_m != "High") & (y_m == 1)).sum() / hours})
        rows.append({"view": view, "attack type": capture, **r})
per_type = pd.DataFrame(rows).set_index(["view", "attack type"])
for view in ["all", "unseen"]:  # every attack type weighs the same in the average
    per_type.loc[(view, "average across types"), :] = per_type.loc[view].mean(numeric_only=True)
per_type.to_csv(OUT / f"{TRACK}_per_attack_type.csv")
per_type[["attack segments", "attacks blocked", "attacks called benign", "Medium", "block precision", "Low precision",
          "false blocks per hour", "attacks passed per hour"]].round(4)

In [ ]:
# Drift: four time slices of each capture's test period
rank = pd.Series(start_of).groupby(capture_of).rank(pct=True).to_numpy()
slice_of = np.minimum((rank * 4).astype(int), 3) + 1
drift = pd.DataFrame([{"slice": s, **known_metrics(out[slice_of == s], main["y"]["test"][slice_of == s])} for s in range(1, 5)]).set_index("slice")
drift.to_csv(OUT / f"{TRACK}_drift.csv")
drift[["segments", "coverage", "attacks blocked", "block precision", "Low precision", "Medium", "OOD"]].round(4)

#### Time to a decision

How much traffic time passes before an answer: the seconds between a segment's first packet and the packet its answer
came at (from the packets' inter-arrival times), for each decision packet K. "Blocks" are High answers on attack
segments; "any answer" covers every segment. Compute time is separate (deployment cost below).

In [ ]:
iat_s = np.expm1(np.array(X[parts["test"], :, kl.COL["iat"]], dtype=np.float64)) / 1e6  # seconds since the flow's previous packet
iat_s[:, 0] = 0  # counted from the segment's first packet
elapsed = np.cumsum(iat_s, axis=1)
rows = []
for name, u in units.items():
    y = u["y"]["test"]
    for K in K_REPORT:
        out, packets = decide(u, "test", K)
        t = elapsed[np.arange(len(out)), packets - 1]
        for answers, m in [("blocks", (out == "High") & (y == 1)), ("any answer", np.ones(len(out), bool))]:
            rows.append({"model": name, "K": K, "answers": answers, "segments": int(m.sum()),
                         "packets, mean": packets[m].mean() if m.any() else np.nan,
                         "seconds, median": np.median(t[m]) if m.any() else np.nan,
                         "seconds, 90th percentile": np.quantile(t[m], 0.9) if m.any() else np.nan})
decision_time = pd.DataFrame(rows).set_index(["model", "K", "answers"])
decision_time.to_csv(OUT / f"{TRACK}_decision_time.csv")
decision_time.round(4)

#### Forced answers and ceilings on the same rows

A ceiling bounds a model only when identical rows are identical *in what the model reads*. Each model has its own key:
"headers" holds exactly the inputs of the model without TTL, destination port and IAT, and "all inputs" those of the
main model (and of the baselines, which read the same inputs), so each model's ceilings on its own key are upper bounds
for it. Both keys are shown for every model. "above own ceiling" flags a forced accuracy above the MaxAcc of the model's
own key on the same rows, which should not happen: the only way it can is the IAT, held in whole microseconds by the
key (data/results/iat_resolution.csv counts the gaps that are not).

In [ ]:
PREFIXES = [1, 2, 3, 5, 10, 20, 50]
rows = []
for name, u in units.items():
    for p in ("val", "test"):
        idx = parts[p]
        for view, m in [("all", np.ones(len(idx), bool)), ("unseen", u["unseen"][p])]:
            for k in PREFIXES:
                pred = (at_prefix(u["prob"][p], u["n"][p], k)[m] >= 0.5).astype(int)
                r = {"model": name, "split": p, "view": view, "k": k, **scores(u["y"][p][m], pred)}
                for key, H in [("all inputs", prefix_all), ("headers", prefix_header)]:
                    c = ceiling(H[idx[m], k - 1], u["y"][p][m])
                    r.update({f"{cname}, {key}": value for cname, value in c.items() if cname != "rows"})
                r["own key"] = u["key"]
                r["above own ceiling"] = bool(r["accuracy"] > r[f"MaxAcc, {u['key']}"] + 1e-12)
                r["rows"] = int(m.sum())
                rows.append(r)
forced = pd.DataFrame(rows).set_index(["model", "split", "view", "k"])
forced.to_csv(OUT / f"{TRACK}_forced_and_ceilings.csv")
if forced["above own ceiling"].any():
    print(f"WARNING: {int(forced['above own ceiling'].sum())} rows of {TRACK}_forced_and_ceilings.csv score above their own "
          f"ceiling; see iat_resolution.csv", flush=True)
no_testbed = selection["models"]["without TTL, destination port and IAT"][0]
# The model without the testbed group, against its exact ceilings (key "headers")
forced.loc[(no_testbed, "test"), ["rows", "accuracy", "MaxAcc, headers", "macro F1", "MaxF1, headers", "FPR",
                                  "minimum FPR, headers", "missed attacks", "minimum missed attacks, headers"]].round(4)

In [ ]:
# The main model, against its exact ceilings (key "all inputs")
forced.loc[(main["model"], "test"), ["rows", "accuracy", "MaxAcc, all inputs", "macro F1", "MaxF1, all inputs", "FPR",
                                     "minimum FPR, all inputs", "missed attacks", "minimum missed attacks, all inputs"]].round(4)

#### Calibration on test

Temperature scaling divides the logit by one temperature per range of packets seen, fitted on the first validation half
and capped at T >= 1 so that calibration never makes the model more confident. Each model is scored with the
temperatures it uses (capped) and, for comparison, with the fitted ones (uncapped), at decision packets 1, 3, 10 and 50.
ECE: 15 equal-width bins of P(attack), the gap between mean P(attack) and the share of attacks in each bin, weighted by
the bin's size. The reliability bins at K = 50 are saved for the reliability diagram.

In [ ]:
def calibration(p, y, bins=15):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    which = np.minimum((p * bins).astype(int), bins - 1)
    ece = sum(abs(p[which == b].mean() - y[which == b].mean()) * (which == b).mean() for b in range(bins) if (which == b).any())
    return {"ECE": ece, "Brier": np.mean((p - y) ** 2), "NLL": -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))}


fitted_T = pd.DataFrame({name: dict(zip(BUCKETS, u["fitted temperatures"][[list(STEP_BUCKET).index(b) for b in range(len(BUCKETS))]]))
                         for name, u in units.items()}).T
print("fitted temperatures (before the cap), per range of packets seen:")
print(fitted_T.round(3))
rows, reliability = [], []
for name, u in units.items():
    for reading, T in [("capped (used)", u["temperatures"]), ("fitted, uncapped", u["fitted temperatures"])]:
        prob = p_attack(u["z"]["test"], T)
        for view, m in [("all", np.ones(len(u["y"]["test"]), bool)), ("unseen", u["unseen"]["test"])]:
            for k in [1, 3, 10, L]:
                p, y = at_prefix(prob, u["n"]["test"], k)[m], u["y"]["test"][m]
                rows.append({"model": name, "temperature": reading, "view": view, "k": k, **calibration(p, y)})
                if k == L and view == "all":
                    which = np.minimum((p * 15).astype(int), 14)
                    reliability += [{"model": name, "temperature": reading, "bin": b, "segments": int((which == b).sum()),
                                     "mean P(attack)": p[which == b].mean(), "attack share": y[which == b].mean()}
                                    for b in range(15) if (which == b).any()]
calib = pd.DataFrame(rows).set_index(["model", "temperature", "view", "k"])
calib.to_csv(OUT / f"{TRACK}_calibration.csv")
pd.DataFrame(reliability).to_csv(OUT / f"{TRACK}_reliability.csv", index=False)
calib.xs("all", level="view").round(4)

Does the cap change the answers? With the uncapped temperatures, the High/Low cut-offs are retuned on the second
validation half (same precision targets, same OOD flags) and the test answers are compared with the ones in use.

In [ ]:
rows = []
for name, u in units.items():
    prob = {p: p_attack(u["z"][p], u["fitted temperatures"]) for p in ("thr", "test")}
    for K in [1, 3, 10, L]:
        flags = u["tuned"][K]["flags"]
        t = tune_targets(prob["thr"], flags["thr"], u["n"]["thr"], u["y"]["thr"], K)
        uncapped = route(prob["test"], flags["test"], u["n"]["test"], t["tau_attack"], t["tau_benign"], K)
        for reading, (out, packets) in [("capped (used)", decide(u, "test", K)), ("fitted, uncapped", uncapped)]:
            for view, m in [("all", np.ones(len(out), bool)), ("unseen", u["unseen"]["test"])]:
                rows.append({"model": name, "temperature": reading, "K": K, "view": view,
                             **known_metrics(out[m], u["y"]["test"][m], packets[m])})
cap_answers = pd.DataFrame(rows).set_index(["model", "temperature", "K", "view"])
cap_answers.to_csv(OUT / f"{TRACK}_cap_answers.csv")
cap_answers.xs("all", level="view")[["coverage", "attacks blocked", "block precision", "attacks called benign", "Low precision",
                                     "Medium", "mean packets to block"]].round(4)

#### Held-out captures: Mirai (another network) and ARP MitM (same network)

On held-out traffic the only correct answer is "I don't know" (Medium or OOD); any High or Low answer is a failure, even
when it matches the label. Mirai's labels are time-based (every packet from 121,622 on counts as attack), so its groups
are read as before and during infection.

In [ ]:
group_of = held.loc[parts["held"], "group"].to_numpy()
cap_of = held.loc[parts["held"], "capture"].to_numpy()
label_of = main["y"]["held"]
rows = []
for name, u in units.items():
    out, _ = decide(u, "held", L)
    for view, vm in [("all", np.ones(len(out), bool)), ("unseen", u["unseen"]["held"])]:
        for capture in ["Mirai", "ARP MitM"]:
            for lab, desc in [(0, "before infection" if capture == "Mirai" else "benign"), (1, "during infection" if capture == "Mirai" else "attack")]:
                m = vm & (cap_of == capture) & (label_of == lab)
                o = out[m]
                rows.append({"model": name, "view": view, "traffic": f"{capture}, {desc}", "segments": int(m.sum()),
                             "failure rate": np.isin(o, ["High", "Low"]).mean(), **{a: (o == a).mean() for a in ["High", "Low", "Medium", "OOD"]}})
held_table = pd.DataFrame(rows).set_index(["model", "view", "traffic"])
held_table.to_csv(OUT / f"{TRACK}_held_out.csv")
held_table.loc[main["model"]].round(4)

In [ ]:
# OOD separation: AUROC of each detector's running mean, known test packets against held-out packets, per prefix bucket
real_t, real_h = real_steps(main["n"]["test"]), real_steps(main["n"]["held"])
rows = []
for detector in kl.DETECTORS:
    s = {p: main["mean"][p][detector] for p in ("test", "held")}
    for group in pd.unique(group_of):
        for b, bucket in enumerate(BUCKETS):
            inb = (STEP_BUCKET == b)[None, :]
            known_s = s["test"][real_t & inb]
            new_s = s["held"][(group_of == group)[:, None] & real_h & inb]
            if len(new_s):
                rows.append({"detector": detector, "held-out": group, "bucket": bucket,
                             "AUROC": roc_auc_score(np.r_[np.zeros(len(known_s)), np.ones(len(new_s))], np.r_[known_s, new_s])})
ood_auc = pd.DataFrame(rows).set_index(["detector", "held-out", "bucket"])["AUROC"].unstack("bucket")[list(BUCKETS)]
ood_auc.to_csv(OUT / f"{TRACK}_ood_auroc.csv")
ood_auc.round(3)

Which detector flags what, at K = 50: the share of segments flagged by the forest only, Mahalanobis only, both, and
either (the OOD answer). Known test traffic shows the cost, the held-out captures what each detector adds.

In [ ]:
rows = []
for name, u in units.items():
    t = u["tuned"][L]
    for p, groups in [("test", np.where(u["y"]["test"] == 1, "known test, attack", "known test, benign")), ("held", group_of)]:
        live = real_steps(u["n"][p])
        by = {d: ((u["mean"][p][d] > np.asarray(t["ood limits"][d])[STEP_BUCKET][None, :]) & live).any(axis=1) for d in kl.DETECTORS}
        for g in pd.unique(groups):
            m = groups == g
            f, mh = by["forest"][m], by["mahalanobis"][m]
            rows.append({"model": name, "traffic": g, "segments": int(m.sum()), "forest only": (f & ~mh).mean(),
                         "Mahalanobis only": (mh & ~f).mean(), "both": (f & mh).mean(), "either": (f | mh).mean()})
detector_overlap = pd.DataFrame(rows).set_index(["model", "traffic"])
detector_overlap.to_csv(OUT / f"{TRACK}_ood_detectors.csv")
detector_overlap.round(4)

#### Baselines: k-NN and XGBoost against the GRU

One section, one training per model: k-NN (L1 distance, 15 neighbours, share of attack among them) and weighted
XGBoost, at each checkpoint k on the first k packets of the main model's inputs. Every model is read at the
same checkpoints with the OOD gate off, gets its High/Low cut-offs from the same precision targets on the same rows of
the second validation half, and is scored on the same 15,000 test rows. Forced answers (P(attack) >= 0.5) sit next to the
ceilings on those rows; the H/M/L reading is the main comparison. These models read the main model's inputs, so the
ceilings are on the "all inputs" key and are upper bounds for every one of them ("above own ceiling" flags a breach).

In [ ]:
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBClassifier

CHECKS, EVAL_ROWS, KNN_TRAIN, NEIGHBOURS = [1, 2, 3, 5, 10, 20, L], 15_000, 40_000, 15
rng = np.random.default_rng(SEED)
pick = {p: np.sort(rng.choice(len(parts[p]), min(EVAL_ROWS, len(parts[p])), replace=False)) for p in ("thr", "test")}
cols = main["columns"]
cache = OUT / f"{TRACK}_baseline_probs.npz"
# What the cached probabilities were computed from: the settings, the rows (training, evaluated), the inputs read and
# their standardisation, and the labels
baseline_fingerprint = {"training version": kl.TRAINING_VERSION, "checks": CHECKS, "evaluated rows": EVAL_ROWS, "k-NN training rows": KNN_TRAIN, "neighbours": NEIGHBOURS,
                        "xgboost": "n_estimators=200, max_depth=6, learning_rate=0.1, hist", "seed": SEED,
                        "data": kl.digest(part["train"], parts["thr"][pick["thr"]], parts["test"][pick["test"]], np.asarray(cols),
                                          stats[0], stats[1], y_all[part["train"]], segments["hash_all"].to_numpy()[part["train"]])}
if cache.exists():
    probs = dict(np.load(cache))
    stored = probs.pop("fingerprint", None)
    kl.check_fingerprint(None if stored is None else json.loads(str(stored)), baseline_fingerprint, cache)
else:
    x_tr, y_tr = load(X, part["train"], stats, cols), y_all[part["train"]]
    knn_rows = rng.permutation(len(x_tr))[:KNN_TRAIN]
    x_ev = {p: load(X, parts[p][pick[p]], stats, cols) for p in pick}
    probs = {f"{m}|{p}": np.zeros((len(pick[p]), len(CHECKS))) for m in ["k-NN", "XGBoost"] for p in pick}
    for j, k in enumerate(CHECKS):
        t0 = time.time()
        flat = x_tr[:, :k].reshape(len(x_tr), -1)
        nn = NearestNeighbors(n_neighbors=NEIGHBOURS, metric="manhattan", algorithm="brute", n_jobs=-1).fit(flat[knn_rows])
        xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, tree_method="hist", n_jobs=-1, random_state=SEED,
                            scale_pos_weight=(y_tr == 0).sum() / (y_tr == 1).sum()).fit(flat, y_tr)
        for p, x in x_ev.items():
            f = x[:, :k].reshape(len(x), -1)
            probs[f"k-NN|{p}"][:, j] = y_tr[knn_rows][nn.kneighbors(f, return_distance=False)].mean(axis=1)
            probs[f"XGBoost|{p}"][:, j] = xgb.predict_proba(f)[:, 1]
        print(f"k={k}: {time.time() - t0:.0f}s")
    np.savez(cache, fingerprint=np.array(json.dumps(baseline_fingerprint)), **probs)
for p in pick:  # the GRU read at the same checkpoints
    probs[f"GRU|{p}"] = np.stack([at_prefix(main["prob"][p][pick[p]], main["n"][p][pick[p]], k) for k in CHECKS], axis=1)

rows, forced_rows = [], []
y_ev = {p: main["y"][p][pick[p]] for p in pick}
full = {p: np.full(len(pick[p]), len(CHECKS)) for p in pick}
no_flags = {p: np.zeros((len(pick[p]), len(CHECKS)), bool) for p in pick}
test_idx = parts["test"][pick["test"]]
test_unseen = main["unseen"]["test"][pick["test"]]
for m in ["GRU", "k-NN", "XGBoost"]:
    t = tune_targets(probs[f"{m}|thr"], no_flags["thr"], full["thr"], y_ev["thr"], len(CHECKS))
    out, _ = route(probs[f"{m}|test"], no_flags["test"], full["test"], t["tau_attack"], t["tau_benign"], len(CHECKS))
    for view, vm in [("all", np.ones(len(out), bool)), ("unseen", test_unseen)]:
        rows.append({"model": m, "view": view, "tau_attack": t["tau_attack"], "tau_benign": t["tau_benign"],
                     **known_metrics(out[vm], y_ev["test"][vm])})
        for j, k in enumerate(CHECKS):
            r = {"model": m, "view": view, "k": k, **scores(y_ev["test"][vm], (probs[f"{m}|test"][vm, j] >= 0.5).astype(int))}
            c = ceiling(prefix_all[test_idx[vm], k - 1], y_ev["test"][vm])  # key "all inputs": exact for every model here
            forced_rows.append({**r, **{f"{name} (same rows)": value for name, value in c.items()},
                                "above own ceiling": bool(r["accuracy"] > c["MaxAcc"] + 1e-12)})
baselines = pd.DataFrame(rows).set_index(["model", "view"])
baselines_forced = pd.DataFrame(forced_rows).set_index(["model", "view", "k"])
baselines.to_csv(OUT / f"{TRACK}_baselines.csv")
baselines_forced.to_csv(OUT / f"{TRACK}_baselines_forced.csv")
if baselines_forced["above own ceiling"].any():
    print(f"WARNING: {int(baselines_forced['above own ceiling'].sum())} baseline rows score above their ceiling; "
          f"see iat_resolution.csv", flush=True)
baselines[["coverage", "attacks blocked", "block precision", "attacks called benign", "benign called Low", "Low precision",
           "accuracy when answering", "macro F1 when answering", "Medium"]].round(4)

In [ ]:
baselines_forced[["accuracy", "MaxAcc (same rows)", "macro F1", "MaxF1 (same rows)", "FPR", "missed attacks"]].round(4)

#### Main model vs without TTL, destination port and IAT

In [ ]:
roles = {n: role for role, names in selection["models"].items() for n in names}
compare = pd.concat({view: pipeline.xs(("test", L, view), level=["split", "K", "view"])[shown] for view in ["all", "unseen"]},
                    names=["view"]).rename(index=roles, level=1)
compare.to_csv(OUT / f"{TRACK}_testbed_comparison.csv")
compare.round(4)

#### Evasion: perturbing what an attacker controls

Test attack segments with timing jitter (each inter-arrival time multiplied by a factor in [0.8, 1.2]) or padding (up to
32 bytes added to each frame), with no retraining. A block that turns into Medium or OOD fails safe; one that turns into
Low is the dangerous failure.

In [ ]:
rng = np.random.default_rng(SEED)
att = parts["test"][main["y"]["test"] == 1]
raw = np.array(X[att])
mask = raw[:, :, kl.MASK_IDX] > 0
variants = {"none": raw}
v = raw.copy(); iat = np.expm1(v[:, :, kl.COL["iat"]]) * rng.uniform(0.8, 1.2, mask.shape); v[:, :, kl.COL["iat"]] = np.where(mask, np.log1p(iat), 0); variants["IAT jitter +/-20%"] = v
v = raw.copy(); size = np.expm1(v[:, :, kl.COL["frame_len"]]) + rng.integers(0, 33, mask.shape); v[:, :, kl.COL["frame_len"]] = np.where(mask, np.log1p(size), 0); variants["padding up to 32 bytes"] = v
mu, sd = main["stats"]
n_att = lengths[att]
t = main["tuned"][L]
rows = []
for vname, raw_v in variants.items():
    x = raw_v.copy()
    x[:, :, kl.CONT_IDX] = (x[:, :, kl.CONT_IDX] - mu) / sd
    x *= x[:, :, [kl.MASK_IDX]]
    z, h = predict(main["net"], x[:, :, main["columns"]].copy(), with_hidden=True)
    flags = kl.ood_flags(kl.ood_running_means(main["detectors"], h, n_att), t["ood limits"])
    del h
    out, _ = route(p_attack(z, main["temperatures"]), flags, n_att, t["tau_attack"], t["tau_benign"], L)
    rows.append({"perturbation": vname, **{a: (out == a).mean() for a in ["High", "Low", "Medium", "OOD"]}})
evasion = pd.DataFrame(rows).set_index("perturbation")
evasion.to_csv(OUT / f"{TRACK}_evasion.csv")
print(f"{len(att):,} test attack segments; inputs used: {main['groups']}")
evasion.round(4)

#### Deployment cost

Test segments of the main model are classified packet by packet, as a deployment would: the GRU step, both OOD scores
(Isolation Forest, Mahalanobis), their running means against the OOD limits, the temperature and the High/Low cut-offs,
stopping at the answer. Everything runs on one thread. The time per packet covers that whole path, with the share of
each part; the answers are checked against the batch decision rule used in evaluation, so the timed code is the logic
that was measured.

In [ ]:
COST_SEGMENTS = 300  # test segments timed
hidden, layers = GRU_GRID[arch]
net = main["net"].eval()
t_L = main["tuned"][L]
limits = {d: np.asarray(t_L["ood limits"][d]) for d in kl.DETECTORS}
forest = copy.deepcopy(main["detectors"]["forest"]).set_params(n_jobs=1)  # one thread, like the rest
maha = main["detectors"]["mahalanobis"]
PARTS = ["GRU", "Isolation Forest", "Mahalanobis", "decision rule"]


@torch.no_grad()
def classify_stream(x, n, K=L):
    """One segment, one packet at a time. Returns its answer, the packet it came at and the seconds spent in each part."""
    spent, sums, state = dict.fromkeys(PARTS, 0.0), np.zeros(len(kl.DETECTORS)), None
    last = min(int(n), K)
    for k in range(1, last + 1):
        t0 = time.perf_counter()
        o, state = net.gru(x[None, k - 1:k], state)
        z = float(net.head(o))
        h = o[0].numpy()
        t1 = time.perf_counter()
        score = {"forest": -forest.score_samples(h)[0]}
        t2 = time.perf_counter()
        score["mahalanobis"] = min(float((((h - mu) @ maha["precision"]) * (h - mu)).sum()) for mu in maha["means"])
        t3 = time.perf_counter()
        sums += [score[d] for d in kl.DETECTORS]
        flagged = any(sums[i] / k > limits[d][STEP_BUCKET[k - 1]] for i, d in enumerate(kl.DETECTORS))
        p = 1 / (1 + np.exp(z / main["temperatures"][k - 1]))
        answer = "OOD" if flagged else "High" if p >= t_L["tau_attack"] else None
        if answer is None and k == last:
            answer = "Low" if p <= t_L["tau_benign"] else "Medium"
        t4 = time.perf_counter()
        for part_name, s in zip(PARTS, [t1 - t0, t2 - t1, t3 - t2, t4 - t3]):
            spent[part_name] += s
        if answer:
            return answer, k, spent


torch.set_num_threads(1)
timed = np.sort(np.random.default_rng(SEED).choice(len(parts["test"]), min(COST_SEGMENTS, len(parts["test"])), replace=False))
x_timed = torch.from_numpy(load(X, parts["test"][timed], stats, main["columns"]))
n_timed = main["n"]["test"][timed]
classify_stream(x_timed[0], n_timed[0])  # warm-up, not counted
runs = [classify_stream(x_timed[i], n_timed[i]) for i in range(len(timed))]
torch.set_num_threads(os.cpu_count())
batch_out, batch_packets = decide(main, "test", L)
same = np.mean([(a, k) == (batch_out[j], batch_packets[j]) for (a, k, _), j in zip(runs, timed)])
if same < 0.99:  # the timed path must be the evaluated decision rule
    raise SystemExit(f"streaming answers agree with evaluation on only {same:.1%} of the timed segments")
timed_packets = sum(k for _, k, _ in runs)
spent = {part_name: sum(s[part_name] for _, _, s in runs) for part_name in PARTS}
per_packet_us = sum(spent.values()) / timed_packets * 1e6
# What a streaming deployment keeps per open flow between packets (float32 = 4 bytes)
state_parts = {
    "GRU hidden state (bytes)": layers * hidden * 4,          # one hidden vector per GRU layer
    "OOD running sums (bytes)": len(kl.DETECTORS) * 4,        # one running sum per detector; divided by the count below
    "packet counter (bytes)": 1,                              # packets seen in the current segment, at most L = 50
}
cost = {"inference per packet, one thread (us)": per_packet_us, "packets per second, one thread": 1e6 / per_packet_us,
        **{f"{part_name}, per packet (us)": s / timed_packets * 1e6 for part_name, s in spent.items()},
        "timed segments": len(timed), "timed packets": int(timed_packets), "answers equal to evaluation": float(same),
        **state_parts, "state per flow (bytes)": sum(state_parts.values()),
        "forest trees": main["detectors"]["forest"].n_estimators}
kl.save_json(cost, OUT / f"{TRACK}_deployment_cost.json")
cost

## Stage 4: experiments (per track)

Every experiment uses this track's own rows, cleaning rule, main model inputs and GRU size, and builds its deployable unit
the same way as stage 3 (temperatures, OOD detectors and limits and precision-target cut-offs from its own validation data).
All results are on test (or held-out) rows, in the "all" and "unseen" views ("unseen" on the key "all inputs", since
every model here reads the main model's inputs).

1. **Retraining on OOD abstentions**: does labelling what the classifier flagged teach it something?
2. **Unseen sessions**: leave one capture out (7 runs), and train on February to May, test on August.
3. **Whole flows**: following each flow through the test period, memory reset vs carried across segments.

Every model reads L = 50-packet segments; how early an answer can be given is shown by the decision packet K
(1, 3, 5, 10, 20, 50) in every table, so no model is trained on another segment length.

### Track A

In [ ]:
%reset -f
import gc
import os

gc.collect()
os.environ["TRACK"] = "A"  # the stage reads its track from here

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

import kitsune_lib as kl
from kitsune_lib import (CAPTURES, GRU_GRID, L, OOD_CAPTURES, SEED, build_unit, decide, fit_gru, fit_stats, group_columns,
                         known_metrics, load, load_data, load_gru, load_json, model_key, p_attack, predict, read_track,
                         route, rows_digest, split_index, time_split)

TRACK = os.environ.get("TRACK", "A")
OUT = kl.RESULT_DIR
HIDDEN_TRAIN = 30_000

X, segments, y_all, lengths, _, _ = load_data()
track = read_track(TRACK).set_index("segment", drop=False)
track["capture_order"] = track["capture"].map({c: i for i, c in enumerate(CAPTURES)})
part = split_index(track)
held = pd.read_parquet(kl.TRACK_DIR / "held_out.parquet").set_index("segment", drop=False)
selection = load_json(OUT / f"{TRACK}_selection.json")
arch, groups = selection["arch"], selection["final"]["main"]
cols = group_columns(groups)
seg_hash = segments[kl.KEYS[model_key(groups)]].to_numpy()  # every model here reads the main inputs: key "all inputs"
stats = fit_stats(X, part["train"])


def halves(idx, frame):
    """The validation split cut in two, stratified by capture and class: temperatures, then thresholds."""
    f = frame.loc[idx]
    strata = (f["capture"] + f["attack"].astype(str)).to_numpy()
    counts = pd.Series(strata).map(pd.Series(strata).value_counts()).to_numpy()
    strata = np.where(counts < 2, f["attack"].astype(str).to_numpy(), strata)  # a group of one: stratify by class only
    if pd.Series(strata).value_counts().min() < 2:
        strata = None
    return train_test_split(idx, test_size=0.5, random_state=SEED, stratify=strata)


def views(out, y, unseen_mask, packets=None):
    return {view: known_metrics(out[m], y[m], None if packets is None else packets[m])
            for view, m in [("all", np.ones(len(out), bool)), ("unseen", unseen_mask)]}


cal_idx, thr_idx = halves(part["val"], track)
parts = {"train sample": np.sort(np.random.default_rng(SEED).choice(part["train"], min(HIDDEN_TRAIN, len(part["train"])), replace=False)),
         "cal": np.sort(cal_idx), "thr": np.sort(thr_idx), "test": part["test"], "held": held.index.to_numpy()}
main_net = load_gru(selection["models"]["main"][0], len(cols), *GRU_GRID[arch],
                    rows=rows_digest(TRACK, part["train"], part["val"], cols, stats, segments))
u0 = build_unit(X, main_net, cols, stats, parts, lengths, y_all)
test_unseen = ~np.isin(seg_hash[part["test"]], seg_hash[part["train"]])
print(f"Track {TRACK}: size {arch}, inputs {groups}")

#### 1. Retraining on OOD abstentions

Each held-out capture is cut in half by time. Early-half segments flagged OOD join the training set with their label, the
GRU is retrained from scratch, and the unit is rebuilt. The late half, which neither model saw, shows what changed, and
known test traffic checks that nothing else got worse.

In [ ]:
hm = held.loc[parts["held"]]
early = (hm["start"] <= hm.groupby("capture")["start"].transform("median")).to_numpy()
out, _ = decide(u0, "held")
queued = parts["held"][early & (out == "OOD")]
print(f"early-half OOD abstentions added to training: {len(queued):,}", hm.loc[queued, "group"].value_counts().to_dict())
train_plus = np.r_[part["train"], queued]
net1, _ = fit_gru(f"{TRACK}_{arch}_retrained", load(X, train_plus, stats, cols), y_all[train_plus].astype(np.float32),
                  load(X, part["val"], stats, cols), y_all[part["val"]].astype(np.float32), *GRU_GRID[arch])
u1 = build_unit(X, net1, cols, stats, {**parts, "train sample": np.r_[parts["train sample"], queued]}, lengths, y_all)

rows = []
for name, u in [("before", u0), ("after", u1)]:
    out, packets = decide(u, "test")
    for view, r in views(out, u["y"]["test"], test_unseen, packets).items():
        rows.append({"model": name, "traffic": f"known test, {view}", **r})
    out, _ = decide(u, "held")
    for group in pd.unique(hm["group"]):
        for lab in (0, 1):
            m = (~early) & (hm["group"].to_numpy() == group) & (u["y"]["held"] == lab)
            o = out[m]
            rows.append({"model": name, "traffic": f"{group}, label {lab}, late half", "segments": int(m.sum()),
                         "failure rate": np.isin(o, ["High", "Low"]).mean() if m.any() else np.nan,
                         **{a: (o == a).mean() if m.any() else np.nan for a in ["High", "Low", "Medium", "OOD"]}})
retraining = pd.DataFrame(rows).set_index(["traffic", "model"]).sort_index()
retraining.to_csv(OUT / f"{TRACK}_retraining.csv")
retraining[["segments", "failure rate", "High", "Low", "Medium", "OOD", "attacks blocked", "block precision", "Low precision"]].round(4)

#### 2. Unseen sessions

Each capture is its own recording, so a session can be held out whole, benign and attack together: the session and the
attack are new, the network is the same. The training sessions' rows of this track are split 80/20 by time into training
and validation; the held-out sessions' rows of this track are the test. The GRU size and inputs are this track's, chosen
on data that included every session, which is a small remaining leak.

In [ ]:
known = [c for c in CAPTURES if c not in OOD_CAPTURES]
SESSIONS = {f"without {c}": ([k for k in known if k != c], [c]) for c in known}
SESSIONS["Feb-May to Aug"] = (["Video Injection", "SSDP Flood", "SYN DoS", "SSL Renegotiation"], ["OS Scan", "Fuzzing", "Active Wiretap"])
rows = []
for run, (train_caps, test_caps) in SESSIONS.items():
    t0 = time.time()
    pool = track[track["capture"].isin(train_caps)].sort_values(["capture_order", "attack", "start"], kind="stable").copy()
    pool["split"] = pd.concat([time_split(g, (0.8,)) for _, g in pool.groupby(["capture", "attack"], sort=False)])
    tr = np.sort(pool.loc[pool["split"] == "train", "segment"].to_numpy())
    va = np.sort(pool.loc[pool["split"] == "val", "segment"].to_numpy())
    te = np.sort(track.loc[track["capture"].isin(test_caps), "segment"].to_numpy())
    st = fit_stats(X, tr)
    tag = "".join(ch for ch in run if ch.isalnum())
    net, _ = fit_gru(f"{TRACK}_{arch}_session_{tag}", load(X, tr, st, cols), y_all[tr].astype(np.float32),
                     load(X, va, st, cols), y_all[va].astype(np.float32), *GRU_GRID[arch])
    c, t = halves(va, pool)
    sp = {"train sample": np.sort(np.random.default_rng(SEED).choice(tr, min(HIDDEN_TRAIN, len(tr)), replace=False)),
          "cal": np.sort(c), "thr": np.sort(t), "test": te}
    u = build_unit(X, net, cols, st, sp, lengths, y_all)
    out, packets = decide(u, "test")
    for view, r in views(out, u["y"]["test"], ~np.isin(seg_hash[te], seg_hash[tr]), packets).items():
        rows.append({"run": run, "view": view, **r})
    print(f"{run}: {time.time() - t0:.0f}s")
sessions = pd.DataFrame(rows).set_index(["run", "view"])
sessions.to_csv(OUT / f"{TRACK}_sessions.csv")
sessions[["segments", "coverage", "attacks blocked", "block precision", "attacks called benign", "Low precision", "Medium", "OOD"]].round(4)

#### 3. Whole flows: memory reset vs carried

Each flow is followed through this track's test period (its rows from the start of the test period of each capture), in
order. **reset**: every segment starts from empty memory, as in training. **carried**: each segment starts from the
hidden state the previous segment of the flow left. Same model and unit, not retrained; carried memory is an inference
change only.

In [ ]:
start_of_test = track.loc[part["test"]].groupby("capture")["start"].min()
stream = track[track["start"] >= track["capture"].map(start_of_test)].sort_values(["capture", "flow", "start"], kind="stable").reset_index(drop=True)
stream["flow_key"] = stream.groupby(["capture", "flow"], sort=False).ngroup()
stream["round"] = stream.groupby("flow_key").cumcount()
x_s = load(X, stream["segment"].to_numpy(), stats, cols)
n_s = stream["length"].to_numpy()
y_s = stream["attack"].to_numpy()
hidden, layers = GRU_GRID[arch]


@torch.no_grad()
def run_stream(carry):
    if not carry:
        return predict(main_net, x_s, with_hidden=True)
    z = np.zeros((len(stream), L), np.float32)
    h = np.zeros((len(stream), L, hidden), np.float32)
    state = torch.zeros(layers, stream["flow_key"].max() + 1, hidden)
    keys, rounds = stream["flow_key"].to_numpy(), stream["round"].to_numpy()
    for r in range(rounds.max() + 1):
        at = np.flatnonzero(rounds == r)
        o, last = main_net.gru(torch.from_numpy(x_s[at]), state[:, keys[at]])
        state[:, keys[at]] = last
        z[at], h[at] = main_net.head(o).squeeze(-1).numpy(), o.numpy()
    return z, h


t = u0["tuned"][L]
rows = []
for mode, carry in [("reset", False), ("carried", True)]:
    z, h = run_stream(carry)
    flags = kl.ood_flags(kl.ood_running_means(u0["detectors"], h, n_s), t["ood limits"])
    del h
    out, packets = route(p_attack(z, u0["temperatures"]), flags, n_s, t["tau_attack"], t["tau_benign"], L)
    d = stream[["flow_key", "attack", "length"]].assign(blocked=out == "High", low=out == "Low", packets=packets)
    d["before"] = d.groupby("flow_key")["length"].cumsum() - d["length"]
    per_flow = d.groupby("flow_key").agg(attack=("attack", "max"), blocked=("blocked", "max"), segments=("blocked", "size"))
    r = {"memory": mode, **known_metrics(out, y_s)}
    for name, f in [("all flows", per_flow), ("multi-segment flows", per_flow[per_flow["segments"] > 1])]:
        r[f"{name}: attack"] = int((f["attack"] == 1).sum())
        r[f"{name}: attack ever blocked"] = f.loc[f["attack"] == 1, "blocked"].mean()
        r[f"{name}: benign ever blocked"] = f.loc[f["attack"] == 0, "blocked"].mean()
    a = d[d["attack"] == 1]
    first_block = a[a["blocked"]].groupby("flow_key").first()
    passed = first_block["before"] + first_block["packets"] - a.groupby("flow_key")["before"].first().loc[first_block.index]
    r["attack packets before block, median"], r["attack packets before block, p95"] = passed.median(), passed.quantile(0.95)
    rows.append(r)
whole_flow = pd.DataFrame(rows).set_index("memory")
whole_flow.to_csv(OUT / f"{TRACK}_whole_flow.csv")
whole_flow.T.round(4)

### Track B

In [ ]:
%reset -f
import gc
import os

gc.collect()
os.environ["TRACK"] = "B"  # the stage reads its track from here

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

import kitsune_lib as kl
from kitsune_lib import (CAPTURES, GRU_GRID, L, OOD_CAPTURES, SEED, build_unit, decide, fit_gru, fit_stats, group_columns,
                         known_metrics, load, load_data, load_gru, load_json, model_key, p_attack, predict, read_track,
                         route, rows_digest, split_index, time_split)

TRACK = os.environ.get("TRACK", "A")
OUT = kl.RESULT_DIR
HIDDEN_TRAIN = 30_000

X, segments, y_all, lengths, _, _ = load_data()
track = read_track(TRACK).set_index("segment", drop=False)
track["capture_order"] = track["capture"].map({c: i for i, c in enumerate(CAPTURES)})
part = split_index(track)
held = pd.read_parquet(kl.TRACK_DIR / "held_out.parquet").set_index("segment", drop=False)
selection = load_json(OUT / f"{TRACK}_selection.json")
arch, groups = selection["arch"], selection["final"]["main"]
cols = group_columns(groups)
seg_hash = segments[kl.KEYS[model_key(groups)]].to_numpy()  # every model here reads the main inputs: key "all inputs"
stats = fit_stats(X, part["train"])


def halves(idx, frame):
    """The validation split cut in two, stratified by capture and class: temperatures, then thresholds."""
    f = frame.loc[idx]
    strata = (f["capture"] + f["attack"].astype(str)).to_numpy()
    counts = pd.Series(strata).map(pd.Series(strata).value_counts()).to_numpy()
    strata = np.where(counts < 2, f["attack"].astype(str).to_numpy(), strata)  # a group of one: stratify by class only
    if pd.Series(strata).value_counts().min() < 2:
        strata = None
    return train_test_split(idx, test_size=0.5, random_state=SEED, stratify=strata)


def views(out, y, unseen_mask, packets=None):
    return {view: known_metrics(out[m], y[m], None if packets is None else packets[m])
            for view, m in [("all", np.ones(len(out), bool)), ("unseen", unseen_mask)]}


cal_idx, thr_idx = halves(part["val"], track)
parts = {"train sample": np.sort(np.random.default_rng(SEED).choice(part["train"], min(HIDDEN_TRAIN, len(part["train"])), replace=False)),
         "cal": np.sort(cal_idx), "thr": np.sort(thr_idx), "test": part["test"], "held": held.index.to_numpy()}
main_net = load_gru(selection["models"]["main"][0], len(cols), *GRU_GRID[arch],
                    rows=rows_digest(TRACK, part["train"], part["val"], cols, stats, segments))
u0 = build_unit(X, main_net, cols, stats, parts, lengths, y_all)
test_unseen = ~np.isin(seg_hash[part["test"]], seg_hash[part["train"]])
print(f"Track {TRACK}: size {arch}, inputs {groups}")

#### 1. Retraining on OOD abstentions

Each held-out capture is cut in half by time. Early-half segments flagged OOD join the training set with their label, the
GRU is retrained from scratch, and the unit is rebuilt. The late half, which neither model saw, shows what changed, and
known test traffic checks that nothing else got worse.

In [ ]:
hm = held.loc[parts["held"]]
early = (hm["start"] <= hm.groupby("capture")["start"].transform("median")).to_numpy()
out, _ = decide(u0, "held")
queued = parts["held"][early & (out == "OOD")]
print(f"early-half OOD abstentions added to training: {len(queued):,}", hm.loc[queued, "group"].value_counts().to_dict())
train_plus = np.r_[part["train"], queued]
net1, _ = fit_gru(f"{TRACK}_{arch}_retrained", load(X, train_plus, stats, cols), y_all[train_plus].astype(np.float32),
                  load(X, part["val"], stats, cols), y_all[part["val"]].astype(np.float32), *GRU_GRID[arch])
u1 = build_unit(X, net1, cols, stats, {**parts, "train sample": np.r_[parts["train sample"], queued]}, lengths, y_all)

rows = []
for name, u in [("before", u0), ("after", u1)]:
    out, packets = decide(u, "test")
    for view, r in views(out, u["y"]["test"], test_unseen, packets).items():
        rows.append({"model": name, "traffic": f"known test, {view}", **r})
    out, _ = decide(u, "held")
    for group in pd.unique(hm["group"]):
        for lab in (0, 1):
            m = (~early) & (hm["group"].to_numpy() == group) & (u["y"]["held"] == lab)
            o = out[m]
            rows.append({"model": name, "traffic": f"{group}, label {lab}, late half", "segments": int(m.sum()),
                         "failure rate": np.isin(o, ["High", "Low"]).mean() if m.any() else np.nan,
                         **{a: (o == a).mean() if m.any() else np.nan for a in ["High", "Low", "Medium", "OOD"]}})
retraining = pd.DataFrame(rows).set_index(["traffic", "model"]).sort_index()
retraining.to_csv(OUT / f"{TRACK}_retraining.csv")
retraining[["segments", "failure rate", "High", "Low", "Medium", "OOD", "attacks blocked", "block precision", "Low precision"]].round(4)

#### 2. Unseen sessions

Each capture is its own recording, so a session can be held out whole, benign and attack together: the session and the
attack are new, the network is the same. The training sessions' rows of this track are split 80/20 by time into training
and validation; the held-out sessions' rows of this track are the test. The GRU size and inputs are this track's, chosen
on data that included every session, which is a small remaining leak.

In [ ]:
known = [c for c in CAPTURES if c not in OOD_CAPTURES]
SESSIONS = {f"without {c}": ([k for k in known if k != c], [c]) for c in known}
SESSIONS["Feb-May to Aug"] = (["Video Injection", "SSDP Flood", "SYN DoS", "SSL Renegotiation"], ["OS Scan", "Fuzzing", "Active Wiretap"])
rows = []
for run, (train_caps, test_caps) in SESSIONS.items():
    t0 = time.time()
    pool = track[track["capture"].isin(train_caps)].sort_values(["capture_order", "attack", "start"], kind="stable").copy()
    pool["split"] = pd.concat([time_split(g, (0.8,)) for _, g in pool.groupby(["capture", "attack"], sort=False)])
    tr = np.sort(pool.loc[pool["split"] == "train", "segment"].to_numpy())
    va = np.sort(pool.loc[pool["split"] == "val", "segment"].to_numpy())
    te = np.sort(track.loc[track["capture"].isin(test_caps), "segment"].to_numpy())
    st = fit_stats(X, tr)
    tag = "".join(ch for ch in run if ch.isalnum())
    net, _ = fit_gru(f"{TRACK}_{arch}_session_{tag}", load(X, tr, st, cols), y_all[tr].astype(np.float32),
                     load(X, va, st, cols), y_all[va].astype(np.float32), *GRU_GRID[arch])
    c, t = halves(va, pool)
    sp = {"train sample": np.sort(np.random.default_rng(SEED).choice(tr, min(HIDDEN_TRAIN, len(tr)), replace=False)),
          "cal": np.sort(c), "thr": np.sort(t), "test": te}
    u = build_unit(X, net, cols, st, sp, lengths, y_all)
    out, packets = decide(u, "test")
    for view, r in views(out, u["y"]["test"], ~np.isin(seg_hash[te], seg_hash[tr]), packets).items():
        rows.append({"run": run, "view": view, **r})
    print(f"{run}: {time.time() - t0:.0f}s")
sessions = pd.DataFrame(rows).set_index(["run", "view"])
sessions.to_csv(OUT / f"{TRACK}_sessions.csv")
sessions[["segments", "coverage", "attacks blocked", "block precision", "attacks called benign", "Low precision", "Medium", "OOD"]].round(4)

#### 3. Whole flows: memory reset vs carried

Each flow is followed through this track's test period (its rows from the start of the test period of each capture), in
order. **reset**: every segment starts from empty memory, as in training. **carried**: each segment starts from the
hidden state the previous segment of the flow left. Same model and unit, not retrained; carried memory is an inference
change only.

In [ ]:
start_of_test = track.loc[part["test"]].groupby("capture")["start"].min()
stream = track[track["start"] >= track["capture"].map(start_of_test)].sort_values(["capture", "flow", "start"], kind="stable").reset_index(drop=True)
stream["flow_key"] = stream.groupby(["capture", "flow"], sort=False).ngroup()
stream["round"] = stream.groupby("flow_key").cumcount()
x_s = load(X, stream["segment"].to_numpy(), stats, cols)
n_s = stream["length"].to_numpy()
y_s = stream["attack"].to_numpy()
hidden, layers = GRU_GRID[arch]


@torch.no_grad()
def run_stream(carry):
    if not carry:
        return predict(main_net, x_s, with_hidden=True)
    z = np.zeros((len(stream), L), np.float32)
    h = np.zeros((len(stream), L, hidden), np.float32)
    state = torch.zeros(layers, stream["flow_key"].max() + 1, hidden)
    keys, rounds = stream["flow_key"].to_numpy(), stream["round"].to_numpy()
    for r in range(rounds.max() + 1):
        at = np.flatnonzero(rounds == r)
        o, last = main_net.gru(torch.from_numpy(x_s[at]), state[:, keys[at]])
        state[:, keys[at]] = last
        z[at], h[at] = main_net.head(o).squeeze(-1).numpy(), o.numpy()
    return z, h


t = u0["tuned"][L]
rows = []
for mode, carry in [("reset", False), ("carried", True)]:
    z, h = run_stream(carry)
    flags = kl.ood_flags(kl.ood_running_means(u0["detectors"], h, n_s), t["ood limits"])
    del h
    out, packets = route(p_attack(z, u0["temperatures"]), flags, n_s, t["tau_attack"], t["tau_benign"], L)
    d = stream[["flow_key", "attack", "length"]].assign(blocked=out == "High", low=out == "Low", packets=packets)
    d["before"] = d.groupby("flow_key")["length"].cumsum() - d["length"]
    per_flow = d.groupby("flow_key").agg(attack=("attack", "max"), blocked=("blocked", "max"), segments=("blocked", "size"))
    r = {"memory": mode, **known_metrics(out, y_s)}
    for name, f in [("all flows", per_flow), ("multi-segment flows", per_flow[per_flow["segments"] > 1])]:
        r[f"{name}: attack"] = int((f["attack"] == 1).sum())
        r[f"{name}: attack ever blocked"] = f.loc[f["attack"] == 1, "blocked"].mean()
        r[f"{name}: benign ever blocked"] = f.loc[f["attack"] == 0, "blocked"].mean()
    a = d[d["attack"] == 1]
    first_block = a[a["blocked"]].groupby("flow_key").first()
    passed = first_block["before"] + first_block["packets"] - a.groupby("flow_key")["before"].first().loc[first_block.index]
    r["attack packets before block, median"], r["attack packets before block, p95"] = passed.median(), passed.quantile(0.95)
    rows.append(r)
whole_flow = pd.DataFrame(rows).set_index("memory")
whole_flow.to_csv(OUT / f"{TRACK}_whole_flow.csv")
whole_flow.T.round(4)

## Stage 5: stateful GRU (per track)

So far every segment starts from empty memory, so packet 51 of a flow looks like new traffic. Here the GRU is **trained**
to carry its hidden state from one segment of a flow to the next; gradients stop at each segment boundary, and a new flow
starts empty. The **reset** model is this track's main model (stage 2): same inputs, size, data and loss, without memory.

- **Chains**: every known segment in flow order. When reading (validation, test), memory is never broken by a missing
  segment: for Track B, conflicting segments pass memory along but never count in the scores.
- **Training**: the track's training segments only (for Track B, conflicting segments are skipped, so memory passes from
  the previous training segment of the flow). At the start of every epoch one pass in flow order, without training,
  computes the state each training segment starts from with the current weights (zeros only at a flow's start); the
  epoch then trains on shuffled batches from those states. Shuffling keeps batches mixed across captures and classes,
  since some attacks appear only late in a capture; the price is that a starting state is up to one epoch old (computed
  with the weights at the epoch's start). Early stopping on the validation segments, read in flow order with memory
  carried through training and validation.
- **Reading**: on validation and test, memory runs through every earlier segment of the flow, as it would in deployment.
- **Decisions**: temperatures on the first validation half, precision-target cut-offs on the second, OOD gate off for
  both models so that only memory differs. Results on validation and test, in the "all" and "unseen" views, plus whole flows.

### Track A

In [ ]:
%reset -f
import gc
import os

gc.collect()
os.environ["TRACK"] = "A"  # the stage reads its track from here

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn as tnn

import kitsune_lib as kl
from kitsune_lib import (BATCH, GRU_GRID, K_REPORT, L, MAX_EPOCHS, PATIENCE, SEED, OnlineGRU, ceiling,
                         fit_stats, fit_temperatures, group_columns, known_metrics, load, load_data, load_gru, load_json, p_attack,
                         read_track, real_steps, route, rows_digest, scores, at_prefix, tune_targets)

TRACK = os.environ.get("TRACK", "A")
OUT = kl.RESULT_DIR

X, segments, y_all, lengths, _, prefix_all = load_data()
seg_hash = segments["hash_all"].to_numpy()  # both models read the main inputs: "unseen" on the key "all inputs"
selection = load_json(OUT / f"{TRACK}_selection.json")
arch, groups = selection["arch"], selection["final"]["main"]
cols = group_columns(groups)
hidden, layers = GRU_GRID[arch]
mask_at = len(cols) - 1

# Chains: every known segment (Track A keeps them all), in flow order; this track's split, or none if not in the track
chain = read_track("A").drop(columns="split").merge(read_track(TRACK)[["segment", "split"]], on="segment", how="left")
chain["split"] = chain["split"].fillna("not in track")
chain = chain.sort_values(["capture", "flow", "start"], kind="stable").reset_index(drop=True)
chain["flow_key"] = chain.groupby(["capture", "flow"], sort=False).ngroup()
split = chain["split"].to_numpy()
seg = chain["segment"].to_numpy()
keys = chain["flow_key"].to_numpy()
y = chain["attack"].to_numpy()
n = chain["length"].to_numpy()
train_rows = np.flatnonzero(split == "train")
stats = fit_stats(X, np.sort(seg[train_rows]))
x = load(X, seg, stats, cols)
weight = float((y[train_rows] == 0).sum() / (y[train_rows] == 1).sum())
print(f"Track {TRACK}: {len(chain):,} chained segments in {chain['flow_key'].nunique():,} flows;",
      chain["split"].value_counts().to_dict())

#### Training with memory carried

In [ ]:
@torch.no_grad()
def read_flows(net, rows, carry=True):
    """Logits of `rows` (in chain order), each segment starting from the state its flow's previous row left."""
    if not carry:
        return kl.predict(net, x[rows])
    net.eval()
    z = np.zeros((len(rows), L), np.float32)
    rounds = pd.Series(keys[rows]).groupby(keys[rows]).cumcount().to_numpy()
    state = torch.zeros(layers, keys.max() + 1, hidden)
    for r in range(rounds.max() + 1):
        at = np.flatnonzero(rounds == r)
        o, last = net.gru(torch.from_numpy(x[rows[at]]), state[:, keys[rows[at]]])
        state[:, keys[rows[at]]] = last
        z[at] = net.head(o).squeeze(-1).numpy()
    return z


def weighted_loss(z, rows):
    """The training loss (class-weighted, real packets only) on logits z of `rows`."""
    real = real_steps(n[rows])
    benign = (1 - y[rows])[:, None]
    w = real * (1 + (weight - 1) * y[rows][:, None])
    return float(((np.logaddexp(0, z) - benign * z) * w).sum() / w.sum())


def train_stateful(name):
    path = kl.MODEL_DIR / f"{name}.pt"
    torch.manual_seed(SEED)
    net = OnlineGRU(len(cols), hidden, layers)
    # The chains (inputs, labels, lengths, flows) and which of their rows train and validate
    fingerprint = kl.training_fingerprint(OnlineGRU, len(cols), hidden, layers, SEED,
                                          kl.digest(x, y, n, keys, split == "train", split == "val"))  # + TRAINING_VERSION
    if path.exists():
        saved = torch.load(path)
        kl.check_fingerprint(saved.get("fingerprint"), fingerprint, path)
        net.load_state_dict(saved["state"])
        return net
    read_rows = np.flatnonzero(np.isin(split, ["train", "val"]))  # validation read after its flow's training segments
    val_at = split[read_rows] == "val"
    opt = torch.optim.Adam(net.parameters(), lr=kl.LR)
    rng = np.random.default_rng(SEED)
    best, best_state, stale, t0 = np.inf, None, 0, time.time()
    for epoch in range(MAX_EPOCHS):
        # Where every training segment starts, with this epoch's weights: one pass in flow order, no training
        start = kl.chain_start_states(net.gru, x, train_rows, keys[train_rows], layers, hidden)
        net.train()
        for b in np.array_split(rng.permutation(len(train_rows)), len(train_rows) // BATCH + 1):
            rows = train_rows[b]
            steps = int(n[rows].max())
            xb, yb = torch.from_numpy(x[rows, :steps]), torch.from_numpy(y[rows].astype(np.float32))
            o, _ = net.gru(xb, start[:, b])
            logits = net.head(o).squeeze(-1)
            w = xb[:, :, mask_at] * (1 + (weight - 1) * yb.unsqueeze(1))
            loss = (tnn.functional.binary_cross_entropy_with_logits(logits, (1 - yb).unsqueeze(1).expand_as(logits),
                                                                    reduction="none") * w).sum() / w.sum()
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        val_loss = weighted_loss(read_flows(net, read_rows)[val_at], read_rows[val_at])
        print(f"{name}: epoch {epoch + 1}, validation loss {val_loss:.4f}, {time.time() - t0:.0f}s", flush=True)
        if val_loss < best:
            best, stale = val_loss, 0
            best_state = {k: v.clone() for k, v in net.state_dict().items()}
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    net.load_state_dict(best_state)
    seconds = time.time() - t0
    torch.save({"state": best_state, "val_loss": best, "weight": weight, "epochs": epoch + 1, "seconds": seconds,
                "fingerprint": fingerprint}, path)
    times = kl.MODEL_DIR / "training_times.csv"
    pd.DataFrame([{"model": name, "finished": time.strftime("%Y-%m-%d %H:%M:%S"), "epochs": epoch + 1, "seconds": round(seconds),
                   "training segments": len(train_rows), "hidden": hidden, "layers": layers}]
                 ).to_csv(times, mode="a", header=not times.exists(), index=False)
    return net


stateful = train_stateful(f"{TRACK}_{arch}_stateful")
main_net = load_gru(selection["models"]["main"][0], len(cols), hidden, layers,
                    rows=rows_digest(TRACK, np.sort(seg[train_rows]), np.sort(seg[split == "val"]), cols, stats, segments))

#### Results: reset (main model) vs carried (stateful)

Both models are read over the same chains; only the memory differs. Every row reported is one of this track's segments.

In [ ]:
val_rows, test_rows = np.flatnonzero(split == "val"), np.flatnonzero(split == "test")
f = chain.iloc[val_rows]
strata = (f["capture"] + f["attack"].astype(str)).to_numpy()
counts = pd.Series(strata).map(pd.Series(strata).value_counts()).to_numpy()
strata = np.where(counts < 2, f["attack"].astype(str).to_numpy(), strata)  # a group of one: stratify by class only
if pd.Series(strata).value_counts().min() < 2:
    strata = None
cal_rows, thr_rows = (np.sort(r) for r in train_test_split(val_rows, test_size=0.5, random_state=SEED, stratify=strata))
train_hashes = seg_hash[seg[train_rows]]
everything = np.arange(len(chain))

forced, pipeline, flows = [], [], []
for mode, net, carry in [("reset", main_net, False), ("carried", stateful, True)]:
    z_all = read_flows(net, everything, carry)  # memory runs through every earlier segment of the flow
    T, _ = fit_temperatures(z_all[cal_rows], n[cal_rows], y[cal_rows])
    p_all = p_attack(z_all, T)
    no_flags = np.zeros_like(p_all, bool)
    for part_name, rows in [("validation", val_rows), ("test", test_rows)]:
        unseen_m = ~np.isin(seg_hash[seg[rows]], train_hashes)
        for view, m in [("all", np.ones(len(rows), bool)), ("unseen", unseen_m)]:
            r = rows[m]
            for k in [1, 5, 20, 50]:
                pred = (at_prefix(z_all[r], n[r], k) < 0).astype(int)
                # Ceiling on the same rows at the same prefix k, key "all inputs" (the main model's inputs): an upper
                # bound for the reset model; a reference value for the carried model, which also reads the flow's
                # earlier segments and so can score above it.
                c = ceiling(prefix_all[seg[r], k - 1], y[r])
                forced.append({"memory": mode, "split": part_name, "view": view, "k": k, **scores(y[r], pred),
                               "MaxAcc (same rows)": c["MaxAcc"], "MaxF1 (same rows)": c["MaxF1"]})
    for K in K_REPORT:
        t = tune_targets(p_all[thr_rows], no_flags[thr_rows], n[thr_rows], y[thr_rows], K)
        out, packets = route(p_all, no_flags, n, t["tau_attack"], t["tau_benign"], K)
        for part_name, rows in [("validation", val_rows), ("test", test_rows)]:
            unseen_m = ~np.isin(seg_hash[seg[rows]], train_hashes)
            for view, m in [("all", np.ones(len(rows), bool)), ("unseen", unseen_m)]:
                r = rows[m]
                pipeline.append({"memory": mode, "split": part_name, "view": view, "K": K, "tau_attack": t["tau_attack"],
                                 "tau_benign": t["tau_benign"], **known_metrics(out[r], y[r], packets[r])})
        if K == L:  # whole flows through the test period, at the working cut-off
            d = chain.iloc[test_rows][["flow_key", "attack", "length"]].assign(blocked=out[test_rows] == "High",
                                                                                 packets=packets[test_rows])
            d["before"] = d.groupby("flow_key")["length"].cumsum() - d["length"]
            per_flow = d.groupby("flow_key").agg(attack=("attack", "max"), blocked=("blocked", "max"), segments=("blocked", "size"))
            row = {"memory": mode}
            for name, fl in [("all flows", per_flow), ("multi-segment flows", per_flow[per_flow["segments"] > 1])]:
                row[f"{name}: attack"] = int((fl["attack"] == 1).sum())
                row[f"{name}: benign"] = int((fl["attack"] == 0).sum())
                row[f"{name}: attack ever blocked"] = fl.loc[fl["attack"] == 1, "blocked"].mean()
                row[f"{name}: benign ever blocked"] = fl.loc[fl["attack"] == 0, "blocked"].mean()
            a = d[d["attack"] == 1]
            first_block = a[a["blocked"]].groupby("flow_key").first()
            passed = first_block["before"] + first_block["packets"] - a.groupby("flow_key")["before"].first().loc[first_block.index]
            row["attack packets before block, median"] = passed.median()
            row["attack packets before block, p95"] = passed.quantile(0.95)
            flows.append(row)

forced = pd.DataFrame(forced).set_index(["split", "view", "k", "memory"]).sort_index()
pipeline = pd.DataFrame(pipeline).set_index(["split", "view", "K", "memory"]).sort_index()
flows = pd.DataFrame(flows).set_index("memory")
forced.to_csv(OUT / f"{TRACK}_stateful_forced.csv")
pipeline.to_csv(OUT / f"{TRACK}_stateful_pipeline.csv")
flows.to_csv(OUT / f"{TRACK}_stateful_whole_flow.csv")
(forced.loc["test"] * 100).round(2)

In [ ]:
shown = ["segments", "coverage", "attacks blocked", "block precision", "attacks called benign", "Low precision",
         "accuracy when answering", "macro F1 when answering", "Medium", "mean packets to block"]
pipeline.xs(L, level="K")[shown].round(4)

In [ ]:
flows.T.round(4)

### Track B

In [ ]:
%reset -f
import gc
import os

gc.collect()
os.environ["TRACK"] = "B"  # the stage reads its track from here

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn as tnn

import kitsune_lib as kl
from kitsune_lib import (BATCH, GRU_GRID, K_REPORT, L, MAX_EPOCHS, PATIENCE, SEED, OnlineGRU, ceiling,
                         fit_stats, fit_temperatures, group_columns, known_metrics, load, load_data, load_gru, load_json, p_attack,
                         read_track, real_steps, route, rows_digest, scores, at_prefix, tune_targets)

TRACK = os.environ.get("TRACK", "A")
OUT = kl.RESULT_DIR

X, segments, y_all, lengths, _, prefix_all = load_data()
seg_hash = segments["hash_all"].to_numpy()  # both models read the main inputs: "unseen" on the key "all inputs"
selection = load_json(OUT / f"{TRACK}_selection.json")
arch, groups = selection["arch"], selection["final"]["main"]
cols = group_columns(groups)
hidden, layers = GRU_GRID[arch]
mask_at = len(cols) - 1

# Chains: every known segment (Track A keeps them all), in flow order; this track's split, or none if not in the track
chain = read_track("A").drop(columns="split").merge(read_track(TRACK)[["segment", "split"]], on="segment", how="left")
chain["split"] = chain["split"].fillna("not in track")
chain = chain.sort_values(["capture", "flow", "start"], kind="stable").reset_index(drop=True)
chain["flow_key"] = chain.groupby(["capture", "flow"], sort=False).ngroup()
split = chain["split"].to_numpy()
seg = chain["segment"].to_numpy()
keys = chain["flow_key"].to_numpy()
y = chain["attack"].to_numpy()
n = chain["length"].to_numpy()
train_rows = np.flatnonzero(split == "train")
stats = fit_stats(X, np.sort(seg[train_rows]))
x = load(X, seg, stats, cols)
weight = float((y[train_rows] == 0).sum() / (y[train_rows] == 1).sum())
print(f"Track {TRACK}: {len(chain):,} chained segments in {chain['flow_key'].nunique():,} flows;",
      chain["split"].value_counts().to_dict())

#### Training with memory carried

In [ ]:
@torch.no_grad()
def read_flows(net, rows, carry=True):
    """Logits of `rows` (in chain order), each segment starting from the state its flow's previous row left."""
    if not carry:
        return kl.predict(net, x[rows])
    net.eval()
    z = np.zeros((len(rows), L), np.float32)
    rounds = pd.Series(keys[rows]).groupby(keys[rows]).cumcount().to_numpy()
    state = torch.zeros(layers, keys.max() + 1, hidden)
    for r in range(rounds.max() + 1):
        at = np.flatnonzero(rounds == r)
        o, last = net.gru(torch.from_numpy(x[rows[at]]), state[:, keys[rows[at]]])
        state[:, keys[rows[at]]] = last
        z[at] = net.head(o).squeeze(-1).numpy()
    return z


def weighted_loss(z, rows):
    """The training loss (class-weighted, real packets only) on logits z of `rows`."""
    real = real_steps(n[rows])
    benign = (1 - y[rows])[:, None]
    w = real * (1 + (weight - 1) * y[rows][:, None])
    return float(((np.logaddexp(0, z) - benign * z) * w).sum() / w.sum())


def train_stateful(name):
    path = kl.MODEL_DIR / f"{name}.pt"
    torch.manual_seed(SEED)
    net = OnlineGRU(len(cols), hidden, layers)
    # The chains (inputs, labels, lengths, flows) and which of their rows train and validate
    fingerprint = kl.training_fingerprint(OnlineGRU, len(cols), hidden, layers, SEED,
                                          kl.digest(x, y, n, keys, split == "train", split == "val"))  # + TRAINING_VERSION
    if path.exists():
        saved = torch.load(path)
        kl.check_fingerprint(saved.get("fingerprint"), fingerprint, path)
        net.load_state_dict(saved["state"])
        return net
    read_rows = np.flatnonzero(np.isin(split, ["train", "val"]))  # validation read after its flow's training segments
    val_at = split[read_rows] == "val"
    opt = torch.optim.Adam(net.parameters(), lr=kl.LR)
    rng = np.random.default_rng(SEED)
    best, best_state, stale, t0 = np.inf, None, 0, time.time()
    for epoch in range(MAX_EPOCHS):
        # Where every training segment starts, with this epoch's weights: one pass in flow order, no training
        start = kl.chain_start_states(net.gru, x, train_rows, keys[train_rows], layers, hidden)
        net.train()
        for b in np.array_split(rng.permutation(len(train_rows)), len(train_rows) // BATCH + 1):
            rows = train_rows[b]
            steps = int(n[rows].max())
            xb, yb = torch.from_numpy(x[rows, :steps]), torch.from_numpy(y[rows].astype(np.float32))
            o, _ = net.gru(xb, start[:, b])
            logits = net.head(o).squeeze(-1)
            w = xb[:, :, mask_at] * (1 + (weight - 1) * yb.unsqueeze(1))
            loss = (tnn.functional.binary_cross_entropy_with_logits(logits, (1 - yb).unsqueeze(1).expand_as(logits),
                                                                    reduction="none") * w).sum() / w.sum()
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        val_loss = weighted_loss(read_flows(net, read_rows)[val_at], read_rows[val_at])
        print(f"{name}: epoch {epoch + 1}, validation loss {val_loss:.4f}, {time.time() - t0:.0f}s", flush=True)
        if val_loss < best:
            best, stale = val_loss, 0
            best_state = {k: v.clone() for k, v in net.state_dict().items()}
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    net.load_state_dict(best_state)
    seconds = time.time() - t0
    torch.save({"state": best_state, "val_loss": best, "weight": weight, "epochs": epoch + 1, "seconds": seconds,
                "fingerprint": fingerprint}, path)
    times = kl.MODEL_DIR / "training_times.csv"
    pd.DataFrame([{"model": name, "finished": time.strftime("%Y-%m-%d %H:%M:%S"), "epochs": epoch + 1, "seconds": round(seconds),
                   "training segments": len(train_rows), "hidden": hidden, "layers": layers}]
                 ).to_csv(times, mode="a", header=not times.exists(), index=False)
    return net


stateful = train_stateful(f"{TRACK}_{arch}_stateful")
main_net = load_gru(selection["models"]["main"][0], len(cols), hidden, layers,
                    rows=rows_digest(TRACK, np.sort(seg[train_rows]), np.sort(seg[split == "val"]), cols, stats, segments))

#### Results: reset (main model) vs carried (stateful)

Both models are read over the same chains; only the memory differs. Every row reported is one of this track's segments.

In [ ]:
val_rows, test_rows = np.flatnonzero(split == "val"), np.flatnonzero(split == "test")
f = chain.iloc[val_rows]
strata = (f["capture"] + f["attack"].astype(str)).to_numpy()
counts = pd.Series(strata).map(pd.Series(strata).value_counts()).to_numpy()
strata = np.where(counts < 2, f["attack"].astype(str).to_numpy(), strata)  # a group of one: stratify by class only
if pd.Series(strata).value_counts().min() < 2:
    strata = None
cal_rows, thr_rows = (np.sort(r) for r in train_test_split(val_rows, test_size=0.5, random_state=SEED, stratify=strata))
train_hashes = seg_hash[seg[train_rows]]
everything = np.arange(len(chain))

forced, pipeline, flows = [], [], []
for mode, net, carry in [("reset", main_net, False), ("carried", stateful, True)]:
    z_all = read_flows(net, everything, carry)  # memory runs through every earlier segment of the flow
    T, _ = fit_temperatures(z_all[cal_rows], n[cal_rows], y[cal_rows])
    p_all = p_attack(z_all, T)
    no_flags = np.zeros_like(p_all, bool)
    for part_name, rows in [("validation", val_rows), ("test", test_rows)]:
        unseen_m = ~np.isin(seg_hash[seg[rows]], train_hashes)
        for view, m in [("all", np.ones(len(rows), bool)), ("unseen", unseen_m)]:
            r = rows[m]
            for k in [1, 5, 20, 50]:
                pred = (at_prefix(z_all[r], n[r], k) < 0).astype(int)
                # Ceiling on the same rows at the same prefix k, key "all inputs" (the main model's inputs): an upper
                # bound for the reset model; a reference value for the carried model, which also reads the flow's
                # earlier segments and so can score above it.
                c = ceiling(prefix_all[seg[r], k - 1], y[r])
                forced.append({"memory": mode, "split": part_name, "view": view, "k": k, **scores(y[r], pred),
                               "MaxAcc (same rows)": c["MaxAcc"], "MaxF1 (same rows)": c["MaxF1"]})
    for K in K_REPORT:
        t = tune_targets(p_all[thr_rows], no_flags[thr_rows], n[thr_rows], y[thr_rows], K)
        out, packets = route(p_all, no_flags, n, t["tau_attack"], t["tau_benign"], K)
        for part_name, rows in [("validation", val_rows), ("test", test_rows)]:
            unseen_m = ~np.isin(seg_hash[seg[rows]], train_hashes)
            for view, m in [("all", np.ones(len(rows), bool)), ("unseen", unseen_m)]:
                r = rows[m]
                pipeline.append({"memory": mode, "split": part_name, "view": view, "K": K, "tau_attack": t["tau_attack"],
                                 "tau_benign": t["tau_benign"], **known_metrics(out[r], y[r], packets[r])})
        if K == L:  # whole flows through the test period, at the working cut-off
            d = chain.iloc[test_rows][["flow_key", "attack", "length"]].assign(blocked=out[test_rows] == "High",
                                                                                 packets=packets[test_rows])
            d["before"] = d.groupby("flow_key")["length"].cumsum() - d["length"]
            per_flow = d.groupby("flow_key").agg(attack=("attack", "max"), blocked=("blocked", "max"), segments=("blocked", "size"))
            row = {"memory": mode}
            for name, fl in [("all flows", per_flow), ("multi-segment flows", per_flow[per_flow["segments"] > 1])]:
                row[f"{name}: attack"] = int((fl["attack"] == 1).sum())
                row[f"{name}: benign"] = int((fl["attack"] == 0).sum())
                row[f"{name}: attack ever blocked"] = fl.loc[fl["attack"] == 1, "blocked"].mean()
                row[f"{name}: benign ever blocked"] = fl.loc[fl["attack"] == 0, "blocked"].mean()
            a = d[d["attack"] == 1]
            first_block = a[a["blocked"]].groupby("flow_key").first()
            passed = first_block["before"] + first_block["packets"] - a.groupby("flow_key")["before"].first().loc[first_block.index]
            row["attack packets before block, median"] = passed.median()
            row["attack packets before block, p95"] = passed.quantile(0.95)
            flows.append(row)

forced = pd.DataFrame(forced).set_index(["split", "view", "k", "memory"]).sort_index()
pipeline = pd.DataFrame(pipeline).set_index(["split", "view", "K", "memory"]).sort_index()
flows = pd.DataFrame(flows).set_index("memory")
forced.to_csv(OUT / f"{TRACK}_stateful_forced.csv")
pipeline.to_csv(OUT / f"{TRACK}_stateful_pipeline.csv")
flows.to_csv(OUT / f"{TRACK}_stateful_whole_flow.csv")
(forced.loc["test"] * 100).round(2)

In [ ]:
shown = ["segments", "coverage", "attacks blocked", "block precision", "attacks called benign", "Low precision",
         "accuracy when answering", "macro F1 when answering", "Medium", "mean packets to block"]
pipeline.xs(L, level="K")[shown].round(4)

In [ ]:
flows.T.round(4)

## Stage 6: uncertainty with MC dropout (chosen model only)

The chosen model (track from `TRACK`, model from `ROLE`: `main` or `no_testbed`) is retrained with the same inputs, size,
data and loss, plus dropout (p = 0.2) between the GRU layers and before the output. MC dropout keeps dropout on at
inference and reads each segment T = 20 times; it is an approximation of a Bayesian neural network.

**Entropy split**, per packet, in bits (binary entropy H, at most 1):
- total = H(mean over passes of P(attack))
- aleatoric = mean over passes of H(P(attack)) (noise the model expects in the data)
- epistemic = total - aleatoric (mutual information: how much the passes disagree, a lack of knowledge)

**Calibration** stays temperature scaling: one temperature per prefix bucket, refitted on the first validation half so
that the *average over passes* is calibrated (capped, T >= 1). The OOD gate (forest OR Mahalanobis) and the decision
rule are unchanged; the gate reads the hidden states of a normal pass without dropout. The High/Low cut-offs are
retuned on the second validation half for the averaged probability.

**Questions**: (1) does averaging over passes change the pipeline results or the calibration? (2) is the epistemic part
higher on the held-out captures, and on confident wrong answers, than on correct ones? It is measured here; it does not
enter the decision rule.

### Track A

In [ ]:
%reset -f
import gc
import os

gc.collect()
os.environ["TRACK"] = "A"  # the stage reads its track from here

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import kitsune_lib as kl
from kitsune_lib import (BUCKETS, GRU_GRID, K_REPORT, L, SEED, STEP_BUCKET, at_prefix, fit_gru, fit_stats, group_columns,
                         known_metrics, load, load_data, load_json, p_attack, read_track, real_steps, route,
                         running_mean, split_index, tune_targets)

TRACK = os.environ.get("TRACK", "A")
ROLE = os.environ.get("ROLE", "no_testbed")
OUT = kl.RESULT_DIR
P_DROP, PASSES = 0.2, 20
HIDDEN_TRAIN = 30_000
ROLES = {"main": "main", "no_testbed": "without TTL, destination port and IAT"}

X, segments, y_all, lengths, _, _ = load_data()
track = read_track(TRACK).set_index("segment", drop=False)
part = split_index(track)
held = pd.read_parquet(kl.TRACK_DIR / "held_out.parquet").set_index("segment", drop=False)
selection = load_json(OUT / f"{TRACK}_selection.json")
arch = selection["arch"]
groups = selection["final"][ROLES[ROLE]]
cols = group_columns(groups)
print(f"Track {TRACK}, model {ROLE}, size {arch}, groups: {groups}")

# The same two validation halves as stage 3
v = track.loc[part["val"]]
strata = (v["capture"] + v["attack"].astype(str)).to_numpy()
counts = pd.Series(strata).map(pd.Series(strata).value_counts()).to_numpy()
strata = np.where(counts < 2, v["attack"].astype(str).to_numpy(), strata)
cal_idx, thr_idx = train_test_split(part["val"], test_size=0.5, random_state=SEED,
                                    stratify=strata if pd.Series(strata).value_counts().min() >= 2 else None)
parts = {"train sample": np.sort(np.random.default_rng(SEED).choice(part["train"], min(HIDDEN_TRAIN, len(part["train"])), replace=False)),
         "cal": np.sort(cal_idx), "thr": np.sort(thr_idx), "test": part["test"], "held": held.index.to_numpy()}
seg_hash = segments[kl.KEYS[kl.model_key(groups)]].to_numpy()  # "unseen" on this model's own key
unseen_of = {p: ~np.isin(seg_hash[i], seg_hash[part["train"]]) for p, i in parts.items()}
group_of = held.loc[parts["held"], "group"].to_numpy()

#### The model with dropout

In [ ]:
class DropoutGRU(kl.OnlineGRU):
    """The same causal GRU with dropout between its layers and before the output."""

    def __init__(self, n_inputs, hidden, layers):
        super().__init__(n_inputs, hidden, layers)
        self.gru = torch.nn.GRU(n_inputs, hidden, num_layers=layers, batch_first=True, dropout=P_DROP if layers > 1 else 0.0)
        self.drop = torch.nn.Dropout(P_DROP)

    def forward(self, x):
        h, _ = self.gru(x)
        return self.head(self.drop(h)).squeeze(-1), h


stats = fit_stats(X, part["train"])
t0 = time.time()
net, _ = fit_gru(f"{TRACK}_{arch}_{ROLE}_mc_dropout", load(X, part["train"], stats, cols), y_all[part["train"]].astype(np.float32),
                 load(X, part["val"], stats, cols), y_all[part["val"]].astype(np.float32), *GRU_GRID[arch],
                 model_class=DropoutGRU)
print(f"trained or loaded in {time.time() - t0:.0f}s")

# Deterministic unit of this model (dropout off): OOD detectors and limits, temperatures, cut-offs
t0 = time.time()
u = kl.build_unit(X, net, cols, stats, parts, lengths, y_all)
print(f"unit built in {time.time() - t0:.0f}s")

#### T passes with dropout on, temperatures for the average, entropy split

In [ ]:
@torch.no_grad()
def mc_logits(model, x, passes=PASSES, seed=SEED):
    """Logits of `passes` reads with dropout on (passes x segments x L)."""
    torch.manual_seed(seed)
    model.train()  # only dropout behaves differently in training mode
    out = np.empty((passes, len(x), x.shape[1]), np.float32)
    for i in range(0, len(x), 4096):
        xb = torch.from_numpy(x[i:i + 4096])
        for t in range(passes):
            out[t, i:i + len(xb)] = model(xb)[0].numpy()
    model.eval()
    return out


def fit_mc_temperatures(zs, lengths_, y):
    """One temperature per prefix bucket so that the mean over passes is calibrated (first validation half), T >= 1."""
    real = real_steps(lengths_)
    attack = np.repeat(y[:, None], L, axis=1)
    fitted = np.ones(len(BUCKETS))
    for b in range(len(BUCKETS)):
        m = real & (STEP_BUCKET == b)[None, :]
        if m.any():
            zb, yb = zs[:, m], attack[m]

            def loss(T):
                p = np.clip(p_attack(zb, T).mean(axis=0), 1e-7, 1 - 1e-7)
                return -np.mean(yb * np.log(p) + (1 - yb) * np.log(1 - p))
            fitted[b] = minimize_scalar(loss, bounds=(0.05, 20), method="bounded").x
    return np.maximum(fitted, 1.0)[STEP_BUCKET], fitted


def entropy(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))


t0 = time.time()
zs = {p: mc_logits(net, load(X, parts[p], stats, cols)) for p in ("cal", "thr", "test", "held")}
T_mc, T_mc_fitted = fit_mc_temperatures(zs["cal"], u["n"]["cal"], u["y"]["cal"])
pbar, unc = {}, {}
for p, z in zs.items():
    probs = p_attack(z, T_mc)
    pbar[p] = probs.mean(axis=0)
    total, aleatoric = entropy(pbar[p]), entropy(probs).mean(axis=0)
    unc[p] = {"total": total, "aleatoric": aleatoric, "epistemic": np.maximum(total - aleatoric, 0)}
    del probs
del zs
print(f"{PASSES} passes in {time.time() - t0:.0f}s")
pd.DataFrame({"deterministic": u["fitted temperatures"][[list(STEP_BUCKET).index(b) for b in range(len(BUCKETS))]],
              "MC average": T_mc_fitted}, index=list(BUCKETS)).rename_axis("bucket (fitted, before the cap)").round(3)

#### Pipeline results: deterministic against the MC average

Same OOD flags for both; the High/Low cut-offs of each are tuned on the second validation half. On held-out traffic any
High or Low answer is a failure.

In [ ]:
mc_tuned = {K: tune_targets(pbar["thr"], u["tuned"][K]["flags"]["thr"], u["n"]["thr"], u["y"]["thr"], K) for K in K_REPORT}
prob = {"deterministic": u["prob"], "MC average": pbar}
tuned = {"deterministic": {K: u["tuned"][K] for K in K_REPORT}, "MC average": mc_tuned}


def decide_with(reading, p, K):
    t = tuned[reading][K]
    return route(prob[reading][p], u["tuned"][K]["flags"][p], u["n"][p], t["tau_attack"], t["tau_benign"], K)


rows, held_rows, answers = [], [], {}
for reading in prob:
    for K in K_REPORT:
        out, packets = decide_with(reading, "test", K)
        for view, m in [("all", np.ones(len(out), bool)), ("unseen", unseen_of["test"])]:
            rows.append({"reading": reading, "K": K, "view": view, **known_metrics(out[m], u["y"]["test"][m], packets[m])})
        out_h, packets_h = decide_with(reading, "held", K)
        for g in pd.unique(group_of):
            o = out_h[group_of == g]
            held_rows.append({"reading": reading, "K": K, "traffic": g, "segments": len(o),
                              "failure rate": np.isin(o, ["High", "Low"]).mean(), **{a: (o == a).mean() for a in kl.ANSWERS}})
        if K == L:
            answers[reading] = {"test": (out, packets), "held": (out_h, packets_h)}
mc_pipeline = pd.DataFrame(rows).set_index(["reading", "K", "view"])
mc_held = pd.DataFrame(held_rows).set_index(["reading", "K", "traffic"])
mc_pipeline.to_csv(OUT / f"{TRACK}_{ROLE}_mc_pipeline.csv")
mc_held.to_csv(OUT / f"{TRACK}_{ROLE}_mc_held_out.csv")
pd.DataFrame({r: t[L] for r, t in tuned.items()}).loc[["tau_attack", "tau_benign", "high target met", "low target met"]]

In [ ]:
shown = ["coverage", "attacks blocked", "block precision", "attacks called benign", "Low precision", "Medium", "OOD",
         "mean packets to block"]
mc_pipeline.xs(L, level="K")[shown].round(4)

In [ ]:
mc_held.xs(L, level="K").round(4)

#### Calibration on test (P(attack) at each segment's last packet)

ECE: 15 equal-width bins of P(attack), the gap between the mean P(attack) and the share of attacks in each bin,
weighted by the bin's size.

In [ ]:
def calibration(p, y, bins=15):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    which = np.minimum((p * bins).astype(int), bins - 1)
    ece = sum(abs(p[which == b].mean() - y[which == b].mean()) * (which == b).mean() for b in range(bins) if (which == b).any())
    return {"ECE": ece, "Brier": np.mean((p - y) ** 2), "NLL": -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))}


y_t, n_t = u["y"]["test"], u["n"]["test"]
rows = []
for reading in prob:
    p_last = at_prefix(prob[reading]["test"], n_t, L)
    for view, m in [("all", np.ones(len(y_t), bool)), ("unseen", unseen_of["test"])]:
        rows.append({"reading": reading, "view": view, **calibration(p_last[m], y_t[m])})
mc_calibration = pd.DataFrame(rows).set_index(["reading", "view"])
mc_calibration.to_csv(OUT / f"{TRACK}_{ROLE}_mc_calibration.csv")
mc_calibration.round(4)

#### Uncertainty by answer and traffic (MC average, K = 50)

Each segment's total, aleatoric and epistemic entropy at the packet its answer came at.

In [ ]:
def at_answer(values, packets):
    return values[np.arange(len(values)), packets - 1]


out_t, pk_t = answers["MC average"]["test"]
out_h, pk_h = answers["MC average"]["held"]
frame = pd.concat([
    pd.DataFrame({"traffic": np.where(y_t == 1, "known test, attack", "known test, benign"), "answer": out_t,
                  "unseen": unseen_of["test"], **{k: at_answer(v, pk_t) for k, v in unc["test"].items()}}),
    pd.DataFrame({"traffic": group_of, "answer": out_h, "unseen": unseen_of["held"],
                  **{k: at_answer(v, pk_h) for k, v in unc["held"].items()}})], ignore_index=True)
by_answer = frame.groupby(["traffic", "answer"])[["total", "aleatoric", "epistemic"]].agg(["mean", "median"])
by_answer.insert(0, "segments", frame.groupby(["traffic", "answer"]).size())
by_answer.to_csv(OUT / f"{TRACK}_{ROLE}_mc_uncertainty_by_answer.csv")
by_answer.round(4)

#### Does the epistemic part find what the model gets wrong?

AUROC of each part (higher on the first group is better; 0.5 is no better than chance), at the answer packet, K = 50:
- **wrong vs right, known test (unseen view)**: High or Low answers that are wrong, against those that are right
- **held-out answered vs right**: High or Low answers on a held-out group (all failures), against right answers on test
- **held-out vs known test, all segments**: the same question the OOD detectors answer; the detectors' running means at
  the last packet are shown alongside

"Caught at 5%" is the share of the first group above the 95th percentile of the right test answers: what a gate on that
part would catch if it were allowed to turn 5% of right answers into "I don't know".

In [ ]:
def separation(score_pos, score_neg):
    if not len(score_pos) or not len(score_neg):
        return {"AUROC": np.nan, "caught at 5%": np.nan, "segments": len(score_pos)}
    return {"AUROC": roc_auc_score(np.r_[np.ones(len(score_pos)), np.zeros(len(score_neg))], np.r_[score_pos, score_neg]),
            "caught at 5%": (score_pos > np.quantile(score_neg, 0.95)).mean(), "segments": len(score_pos)}


known = frame[frame["traffic"].str.startswith("known test")]
said = known[known["answer"].isin(["High", "Low"]) & known["unseen"]]
said_right = (said["answer"] == "High") == (said["traffic"] == "known test, attack")
right, wrong = said[said_right], said[~said_right]
rows = []
for part_name in ["total", "aleatoric", "epistemic"]:
    rows.append({"score": part_name, "comparison": "wrong vs right, known test (unseen)", **separation(wrong[part_name], right[part_name])})
    for g in pd.unique(group_of):
        h = frame[(frame["traffic"] == g) & frame["answer"].isin(["High", "Low"])]
        rows.append({"score": part_name, "comparison": f"{g}, answered vs right test", **separation(h[part_name], right[part_name])})

# Held-out against known test, all segments, at the last packet: running mean of epistemic entropy and of each detector
last = {p: {"epistemic": at_prefix(running_mean(np.where(real_steps(u["n"][p]), unc[p]["epistemic"], 0), u["n"][p]), u["n"][p], L),
            **{d: at_prefix(u["mean"][p][d], u["n"][p], L) for d in kl.DETECTORS}} for p in ("test", "held")}
for score in ["epistemic", *kl.DETECTORS]:
    for g in pd.unique(group_of):
        s = separation(last["held"][score][group_of == g], last["test"][score])
        rows.append({"score": f"{score} (running mean)", "comparison": f"{g} vs known test", **s})
mc_errors = pd.DataFrame(rows).set_index(["comparison", "score"]).sort_index()
mc_errors.to_csv(OUT / f"{TRACK}_{ROLE}_mc_error_detection.csv")
mc_errors.round(4)

## Stage 7: drift monitoring with the population stability index (per track)

The OOD gate judges one segment at a time. A deployment also needs to know when the traffic **as a whole** has moved,
to trigger a review or retraining. The population stability index (PSI) compares the distribution of one variable in a
reference window with a current window:

PSI = sum over bins of (current share - reference share) x ln(current share / reference share)

Bins are deciles of the reference (variables with few values use one bin per value); empty bins get a floor of 0.0001.
Usual reading: below 0.1 stable, 0.1 to 0.25 moderate shift, above 0.25 large shift.

**Variables**: every input field per real packet (raw values, before standardisation), P(attack) at each segment's last
packet, and the mix of answers (High, Low, Medium, OOD at K = 50), from the model given by `ROLE` (default: without TTL,
destination port and IAT).

**Comparisons** (reference -> current): training -> test (the time split); Feb-May captures -> Aug captures (the same
grouping as the stage-4 time shift); training -> each held-out capture (ARP MitM: same network; Mirai: another network).
Each is computed on benign segments only and on all segments: an attack shifts the distribution by itself, so drift of
normal traffic is read on the benign columns.

### Track A

In [ ]:
%reset -f
import gc
import os

gc.collect()
os.environ["TRACK"] = "A"  # the stage reads its track from here

In [ ]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import kitsune_lib as kl
from kitsune_lib import COLUMNS, GRU_GRID, L, MASK_IDX, SEED, at_prefix, load, load_data, load_gru, load_json, p_attack, predict, psi, read_track, route, split_index

TRACK = os.environ.get("TRACK", "A")
ROLE = os.environ.get("ROLE", "no_testbed")
OUT = kl.RESULT_DIR
ROLES = {"main": "main", "no_testbed": "without TTL, destination port and IAT"}
SAMPLE = 30_000  # segments per group, at most
FEB_MAY = ["Video Injection", "SSDP Flood", "SYN DoS", "SSL Renegotiation"]
AUG = ["OS Scan", "Fuzzing", "Active Wiretap"]

X, segments, y_all, lengths, _, _ = load_data()
track = read_track(TRACK).set_index("segment", drop=False)
part = split_index(track)
held = pd.read_parquet(kl.TRACK_DIR / "held_out.parquet").set_index("segment", drop=False)
selection = load_json(OUT / f"{TRACK}_selection.json")
name = selection["models"][ROLES[ROLE]][0]
unit = joblib.load(kl.MODEL_DIR / f"{name}_unit.joblib")
net = load_gru(name, len(unit["columns"]), *GRU_GRID[selection["arch"]],
               rows=kl.rows_digest(TRACK, part["train"], part["val"], unit["columns"], unit["stats"], segments))
kl.check_unit(unit, net.state_dict(), kl.MODEL_DIR / f"{name}_unit.joblib")  # the unit stage 3 built for this model
print(f"Track {TRACK}, model {name}")

#### Groups compared

In [ ]:
rng = np.random.default_rng(SEED)


def sample(idx):
    idx = np.asarray(idx)
    return np.sort(rng.choice(idx, min(SAMPLE, len(idx)), replace=False)) if len(idx) else idx


cap = track["capture"]
groups = {
    "training": part["train"], "test": part["test"],
    "Feb-May captures": track.index[cap.isin(FEB_MAY)].to_numpy(), "Aug captures": track.index[cap.isin(AUG)].to_numpy(),
    **{f"held-out {c}": held.index[held["capture"] == c].to_numpy() for c in kl.OOD_CAPTURES},
}
groups = {g: {"all": sample(i), "benign": sample(i[y_all[i] == 0])} for g, i in groups.items()}
comparisons = [("training", "test"), ("Feb-May captures", "Aug captures")] + [("training", f"held-out {c}") for c in kl.OOD_CAPTURES]
pd.DataFrame({g: {k: len(v) for k, v in d.items()} for g, d in groups.items()}).T

#### PSI

In [ ]:
def packets(idx):
    """Raw input fields of every real packet of these segments."""
    x = np.array(X[idx])
    real = x[:, :, MASK_IDX] > 0
    return pd.DataFrame(x[real], columns=COLUMNS).drop(columns="mask")


def model_view(idx):
    """P(attack) at the last packet and the answer at K = L, from the chosen model's deployable unit."""
    z, h = predict(net, load(X, idx, unit["stats"], unit["columns"]), with_hidden=True)
    n = lengths[idx]
    p = p_attack(z, unit["temperatures"])
    t = unit["thresholds"][L]
    flags = kl.ood_flags(kl.ood_running_means(unit["detectors"], h, n), t["ood limits"])
    out, _ = route(p, flags, n, t["tau_attack"], t["tau_benign"], L)
    return pd.DataFrame({"P(attack)": at_prefix(p, n, L), "answer": out})


views = {g: {k: (packets(i), model_view(i)) for k, i in d.items()} for g, d in groups.items()}
rows = []
for ref, cur in comparisons:
    for k in ["benign", "all"]:
        (rp, rm), (cp, cm) = views[ref][k], views[cur][k]
        r = {"comparison": f"{ref} -> {cur}", "segments": k}
        r.update({c: psi(rp[c], cp[c]) for c in rp.columns})
        r.update({"P(attack)": psi(rm["P(attack)"], cm["P(attack)"]), "answer mix": psi(rm["answer"], cm["answer"])})
        rows.append(r)
table = pd.DataFrame(rows).set_index(["comparison", "segments"])
table.to_csv(OUT / f"{TRACK}_{ROLE}_psi.csv")
table.T.round(3)

Reading: below 0.1 stable, 0.1 to 0.25 moderate, above 0.25 large. Inputs not read by this model (for the model
without the testbed group: TTL, destination port and IAT) are shown for information only.

In [ ]:
level = table.T.apply(lambda s: pd.cut(s, [-np.inf, 0.1, 0.25, np.inf], labels=["stable", "moderate", "large"]))
level

In [ ]:
# Answer mix of each group (shares), to read the "answer mix" PSI
mix = pd.DataFrame({(g, k): v[1]["answer"].value_counts(normalize=True) for g, d in views.items() for k, v in d.items()}).T.fillna(0)
mix.reindex(columns=kl.ANSWERS, fill_value=0).round(4)